
## Learning objectives
By the end of this notebook, you should be able to:
1. represent uncertain parameters as trajectories over time, not just Year-1/Year-40 endpoints;
2. compare static, staged, and flexible policies in the context of the SBB MehrSpur project;
3. define signposts, triggers, lead times, and activation years for major railway expansions;
4. analyse outcome evolution over the full 40-year horizon;
5. identify recurring types of time-series behaviour;
6. identify adaptation tipping points and opportunity points;
7. explore vulnerable futures using PRIM for critical performance targets (e.g., maintaining a 35% PT mode share);
8. design your own adaptive pathway.
## Scope
Costs shown here are simple **undiscounted** sums (investment + annual operating cost) — useful for comparing pathways, but they are *not* Net Present Cost. Discounting and formal appraisal are Notebook 05's job; nothing in this notebook should be read as a final cost comparison.


## Setup
New in this notebook: `code/pathways.py`. Notebooks 01–03 use `code/simulation_engine.py` and `code/policies.py`, which focus on static or simple evaluations. `pathways.py` builds directly on top of them, adding everything needed for adaptive planning: shaped uncertainty trajectories, two independent triggers (Stage 0→1 for the Station Package and Stage 1→2 for the Core Tunnel & 15-minute frequency expansion), and the nine pathways below. It reuses `simulation_engine.simulate_year()` for all underlying transport physics and mode split calculations (car vs. rail/PT, travel times, congestion delays, CO2 emissions) — keeping the transport simulation identical while adding dynamic decision logic.

Like Notebook 03, the exploratory modeling engine (Part 6 onward) is built directly on the **EMA Workbench**: `u_beta_pt`, `u_demand`, and trajectory `shape` choices are registered as `Model` uncertainties, `pathway` (which of the nine policies) as a `lever`, and full 40-year outcome arrays as `TimeSeriesOutcome`s — mirroring the standard pattern for stochastic dynamic systems.

In [ ]:
# Install the shared project requirements into the active notebook kernel
from pathlib import Path
_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
%pip install --quiet -r "$_PROJECT_ROOT/requirements.txt"


In [ ]:

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "code"))
(PROJECT_ROOT / "figures").mkdir(exist_ok=True)
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

import parameters as p
import simulation_engine as m
import pathways as pw
from parameters import N_YEARS, STRUCTURAL_UNCERTAINTIES, PERTURBABLE_PARAMS, NOMINAL_PARAMS

from ema_workbench import (
    Model, RealParameter, CategoricalParameter, ScalarOutcome, TimeSeriesOutcome,
    SequentialEvaluator, perform_experiments, Samplers,
)

try:
    from ema_workbench import Policy            # EMA Workbench 2.x
except ImportError:
    from ema_workbench import Sample as Policy  # EMA Workbench 3.x

from ema_workbench.analysis import feature_scoring, prim

import time as _time; _NB_START = _time.perf_counter()

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)

print("Ready.")
print(f"{len(pw.PATHWAY_NAMES)} pathways defined:", pw.PATHWAY_NAMES)
t1, t2 = pw.TRIGGER_1, pw.TRIGGER_2
print(f"Trigger 1 ({t1['signpost']} > {t1['threshold']:,} trips, "
      f"persistence={t1['persistence']}y, lead time={t1['lead_time']}y) -> activates Stage 1 (Station Package)")
print(f"Trigger 2 ({t2['signpost']} > {t2['threshold']} min, "
      f"persistence={t2['persistence']}y, lead time={t2['lead_time']}y) -> activates Stage 2 (Core Tunnel)")

# Part 1 — Adaptive planning concepts

## Three kinds of policy

- **Static policy** — a single decision, made once, that never changes: build everything now (or never build anything), and stop. `baseline`, `static1`, and `static2` are the three static policies here (Stage 0: Baseline Network, Stage 1: Station Approaches & Hubs (A3, A4, A5), Stage 2: Core Tunnel & Winterthur Hub (A0, A1, A2), held fixed for the full 40-year horizon).
- **Predetermined staged policy** — a committed *multi-phase* plan, scheduled entirely in advance. Every phase's activation year is fixed at the outset, regardless of how the future actually unfolds. `staged1`, `staged2`, and `staged3` are staged in this sense.
- **Flexible policy (adaptive pathway)** — a plan that keeps decisions open and lets implementation depend on how the future actually unfolds, monitored via **triggers**. `flexible1`, `flexible2`, and `flexible3` are the adaptive pathways here.

## The vocabulary of a trigger

| Term | Meaning |
|---|---|
| **Signpost** | The observable time series being monitored (e.g., `avg_tt_min`, `pt_share`, or `congestion_delay_hours`) — the real-world operational metric you track over time. |
| **Trigger threshold** | The critical value the signpost must cross to initiate an action (e.g., travel time exceeding 17.5 minutes or PT trips exceeding 78,000). |
| **Threshold-crossing year** | The first year the signpost crosses the threshold — *before* any persistence requirement is applied. |
| **Decision year** | The year the persistence requirement is satisfied and a formal investment/construction decision is made (may be later than the threshold-crossing year if a sustained trend is required — see Part 4). |
| **Implementation lead time** | How many years it takes to plan, procure, and construct the infrastructure once decided (e.g., environmental approval, tunneling, signaling upgrades). |
| **Activation year** | `decision year + lead time` — the year the new infrastructure or operational stage actually opens for service. |

## Four related but different events

These are easy to blur together — keep them separate:

1. **A trigger being reached** — the signpost has crossed the threshold (possibly only due to short-term fluctuations).
2. **An intervention being decided** — the persistence requirement is satisfied; a binding decision year is locked in.
3. **An intervention becoming operational** — the activation year has arrived; the new rail infrastructure/service is operational.
4. **The current stage becoming insufficient** — the *transport system itself* fails a critical performance requirement, independent of any decision or trigger (Part 12's **adaptation tipping point**).

A trigger firing and the current stage failing are **not the same event** — a well-designed trigger fires *ahead of time* (accounting for lead times) before the tipping point is reached (see Part 14); a poorly designed one fires too late.

## Adaptation tipping points, opportunity points, critical points

- **Adaptation tipping point** — the first year the *current* infrastructure stage can no longer meet the performance target (e.g., corridor travel time exceeds acceptable limits or PT share drops below the 35% target; defined precisely in Part 12).
- **Opportunity point** — the first year upgrading to the *next* stage becomes clearly beneficial, even before the current system fails (Part 13).
- **Critical point** — a year of unusually large change in an outcome trajectory, whatever the cause (Part 15) — including step-changes *caused by* a major tunnel or timetable expansion opening, which is distinct from a system degradation tipping point.


# Part 2 — The nine policies

One baseline plus eight expansion pathways (combining static, predetermined staged, and dynamic adaptive strategies), all defined in `code/pathways.py` (`PATHWAYS`):


In [ ]:
policy_table = pd.DataFrame([
    {
        "policy": key,
        "name": spec["name"],
        "description": spec["description"],
        "initial stage": spec["initial_stage"],
        "Stage 0->1": (spec["to1"]["type"] if spec["to1"] else "never"),
        "Stage 1->2": (spec["to2"]["type"] if spec["to2"] else "never"),
    }
    for key, spec in pw.PATHWAYS.items()
]).set_index("policy")

display(policy_table)


## Visual schedule — static, staged, and flexible policies

- **Static policies** (`baseline`, `static1`, `static2`) remain at their initial stage for the full 40-year horizon.
- **Predetermined staged policies** (`staged1`, `staged2`, `staged3`) transition at fixed, committed calendar years (e.g., Year 10 or Year 30) regardless of how demand or mode share evolves.
- **Flexible policies** (`flexible1`, `flexible2`, `flexible3`) adapt dynamically: transitions to Stage 1 (Station Package) and Stage 2 (Core Tunnel) only occur if monitored signposts cross their thresholds.


In [ ]:
ALL_POLICIES = pw.PATHWAY_NAMES

rng = np.random.default_rng(42)
TRIGGER_YEAR_RANGES = {"to1": (6, 20), "to2": (18, 34)}
TRIGGER_PLACEHOLDERS = {}       # illustrative trigger year, randomised per (policy, transition)
TRIGGER_ALT_PLACEHOLDERS = {}   # a later, delayed alternative -- signpost takes longer, stage still steps up

for key, spec in pw.PATHWAYS.items():
    for transition in ("to1", "to2"):
        rule = spec[transition]
        if rule and rule["type"] == "trigger":
            lo, hi = TRIGGER_YEAR_RANGES[transition]
            primary = int(rng.integers(lo, hi + 1))
            alt = int(rng.integers(primary + 3, min(hi + 8, N_YEARS - 2)))
            TRIGGER_PLACEHOLDERS[(key, transition)] = primary
            TRIGGER_ALT_PLACEHOLDERS[(key, transition)] = alt

POLICY_COLORS = dict(zip(ALL_POLICIES,
    ["#7f7f7f", "#4C78A8", "#E45756", "#54A24B", "#72B7B2", "#B279A2",
     "#F2CF5B", "#FF9DA6", "#9D755D"]))

fig, ax = plt.subplots(figsize=(12, 6))

for i, key in enumerate(ALL_POLICIES):
    spec = pw.PATHWAYS[key]
    flexible = any(rule and rule["type"] == "trigger"
                   for rule in [spec["to1"], spec["to2"]])

    points = [(1, spec["initial_stage"])]
    for transition, stage in [("to1", 1), ("to2", 2)]:
        rule = spec[transition]
        if rule:
            year = rule["year"] if rule["type"] == "fixed" else TRIGGER_PLACEHOLDERS[(key, transition)]
            points.append((year, stage))

    points.sort()
    years = [p[0] for p in points] + [N_YEARS]
    stages = [p[1] + i * 0.035 for p in points] + [points[-1][1] + i * 0.035]

    ax.step(years, stages, where="post", color=POLICY_COLORS[key],
            linewidth=2.5, linestyle="--" if flexible else "-",
            label=spec["name"])

    if flexible:
        alt_points = [(1, spec["initial_stage"])]
        for transition, stage in [("to1", 1), ("to2", 2)]:
            rule = spec[transition]
            if rule:
                year = (rule["year"] if rule["type"] == "fixed"
                        else TRIGGER_ALT_PLACEHOLDERS[(key, transition)])
                alt_points.append((year, stage))
        alt_points.sort()
        alt_years = [p[0] for p in alt_points] + [N_YEARS]
        alt_stages = [p[1] + i * 0.035 for p in alt_points] + [alt_points[-1][1] + i * 0.035]
        ax.step(alt_years, alt_stages, where="post", color=POLICY_COLORS[key],
                linewidth=1.8, linestyle=":", alpha=0.7)

    for transition, marker in [("to1", "T1?"), ("to2", "T2?")]:
        rule = spec[transition]
        if rule and rule["type"] == "trigger":
            year = TRIGGER_PLACEHOLDERS[(key, transition)]
            stage = 1 if transition == "to1" else 2
            ax.text(year, stage + i * 0.035 + 0.05, marker,
                    color=POLICY_COLORS[key], fontsize=8, ha="center")

ax.set_yticks([0, 1, 2])
ax.set_yticklabels(["Stage 0 (Baseline)", "Stage 1 (Station Package)", "Stage 2 (Tunnel + 15-min)"])
ax.set_xlabel("Year")
ax.set_title("All Nine Pathways — Fixed Schedules and Flexible Trigger Logic\n"
             "(Dashed transitions and T1?/T2? positions represent schematic dynamic adaptation years)")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_all_pathway_schedules.png",
            dpi=150, bbox_inches="tight")
plt.show()


## Conceptual pathways diagram

```text
Stage 0 ──────► Stage 1 ──────► Stage 2
        fixed year OR         fixed year OR
        Trigger 1             Trigger 2
        (PT demand)           (highway congestion)
```

- **Fixed transitions** (`staged1/2/3`): predetermined, drawn as a solid arrow with a year attached.
- **Triggered transitions** (`flexible1/2/3`): conditional on Trigger 1 / Trigger 2, drawn as a dashed arrow — *whether* and *when* it fires depends on the future.
- **`static1`/`static2`** skip all arrows — they start directly at Stage 1 / Stage 2 respectively and stay there.
- **`baseline`** takes neither arrow — it stays at Stage 0 for the full horizon.

Part 18 turns this sketch into a real, data-driven pathways map once we know how often each route is actually taken.


# Part 3 — From snapshots to trajectories

Rather than evaluating static parameter snapshots, adaptive planning requires modeling full **40-year trajectories**. Two futures can share the exact same starting point and 40-year endpoint, yet unfold completely differently in between — one experiencing rapid early growth that triggers capacity investments early on, while another grows slowly until late in the horizon. That timing directly dictates when infrastructure decisions must be taken.

We model 40-year trajectories for public transport preference (`u_beta_pt`) and corridor travel demand (`u_demand`) using six trajectory shapes (`code/pathways.py`, `SHAPES`):

**Interpolated & Scenario-Shaped** (deterministic paths to target endpoints):
- **`linear`** — steady, constant-rate annual change.
- **`early`** — rapid early growth followed by an extended plateau.
- **`late`** — slow initial growth accelerating towards the end of the horizon.
- **`logistic`** — classic S-curve adoption pattern with a fast transition window.
- **`almost_flat`** — low-growth trajectory reaching only 20% of the nominal 40-year change.

**Stochastic**:
- **`random_walk`** — a mean-reverting Ornstein-Uhlenbeck random walk around the linear trend. This shape does **not** force a fixed Year-40 endpoint — capturing realistic real-world volatility where the long-term future is genuinely open-ended.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

SHAPE_COLORS = {
    "linear": "steelblue",
    "early": "seagreen",
    "late": "darkorange",
    "logistic": "mediumpurple",
    "random_walk": "crimson",
    "almost_flat": "gray",
}
years = np.arange(1, N_YEARS + 1)
u_demo = 0.85   # illustrative parameter draw

for shape in pw.SHAPES:
    style = "--" if shape == "random_walk" else "-"
    axes[0].plot(years, pw.pt_affinity_trajectory(u_demo, shape), style, color=SHAPE_COLORS[shape],
                 linewidth=2.2, label=shape)
    axes[1].plot(years, pw.demand_growth_trajectory(u_demo, shape), style, color=SHAPE_COLORS[shape],
                 linewidth=2.2, label=shape)

axes[0].set_title(f"PT Preference Multiplier (u_beta_pt), u={u_demo}")
axes[0].set_ylabel("u_beta_pt multiplier")
axes[1].set_title(f"Cumulative Demand Growth (u_demand), u={u_demo}")
axes[1].set_ylabel("Cumulative Demand Growth (fraction)")

for ax in axes:
    ax.set_xlabel("Year")
    ax.legend(title="shape", fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("Same uncertainty draw (u = 0.85) across six trajectory shapes", y=1.03, fontsize=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_trajectory_shapes.png", dpi=150, bbox_inches="tight")
plt.show()


## Fan charts of the sampled uncertainty trajectories

Instead of a single draw, we sample multiple parameter draws across different shapes to visualize the full *spread* and distribution of trajectories for public transport preference (`u_beta_pt`) and corridor demand growth (`u_demand`).


In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from scipy.stats import gaussian_kde

N_PER_SHAPE = 50
rng = np.random.default_rng(4)

# Same uncertainty draws for every shape, making the shape comparison fair
u_beta_draws = rng.uniform(0, 1, N_PER_SHAPE)
u_demand_draws = rng.uniform(0, 1, N_PER_SHAPE)

beta_by_shape = {
    shape: np.array([pw.pt_affinity_trajectory(u, shape) for u in u_beta_draws])
    for shape in pw.SHAPES
}
demand_by_shape = {
    shape: np.array([pw.demand_growth_trajectory(u, shape) for u in u_demand_draws])
    for shape in pw.SHAPES
}

beta_all = np.vstack(list(beta_by_shape.values()))
demand_all = np.vstack(list(demand_by_shape.values()))

def plot_fan(ax, fan, color, line_color="darkgrey", line_width=0.65, line_alpha=0.15):
    p10, p25, p50, p75, p90 = np.percentile(fan, [10, 25, 50, 75, 90], axis=0)

    ax.fill_between(years, p10, p90, color=color, alpha=0.25, zorder=1)
    ax.fill_between(years, p25, p75, color=color, alpha=0.25, zorder=2)

    for trajectory in fan:
        ax.plot(years, trajectory, color=line_color or color,
                alpha=line_alpha, linewidth=line_width, zorder=3)

    ax.plot(years, p50, color=color, linewidth=2.2, zorder=4)
    ax.grid(alpha=0.3)

def plot_density(ax, values, color):
    """Vertical KDE of `values` (e.g. a fan's final-year outcomes), sharing the
    y-axis with the companion fan panel so the two line up directly."""
    lo, hi = values.min(), values.max()
    y_grid = np.linspace(lo, hi, 200)
    density = gaussian_kde(values)(y_grid)

    ax.fill_betweenx(y_grid, 0, density, color=color, alpha=0.35, zorder=2)
    ax.plot(density, y_grid, color=color, linewidth=1.2, zorder=3)

    ax.set_xlim(left=0)
    ax.set_xticks([])
    ax.tick_params(axis="y", left=False, labelleft=False)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)

n_rows = len(pw.SHAPES) + 1
fig = plt.figure(figsize=(17, 3 * n_rows))
gs = fig.add_gridspec(n_rows, 5, width_ratios=[4, 1, 0.6, 4, 1],
                      wspace=0.12, hspace=0.4)

axes = np.empty((n_rows, 2), dtype=object)       # fan panels: col 0 = beta, col 1 = demand
dens_axes = np.empty((n_rows, 2), dtype=object)  # their final-year density panels

for row in range(n_rows):
    ref_beta = axes[0, 0] if row else None
    ref_demand = axes[0, 1] if row else None

    axes[row, 0] = fig.add_subplot(gs[row, 0], sharex=ref_beta, sharey=ref_beta)
    dens_axes[row, 0] = fig.add_subplot(gs[row, 1], sharey=axes[row, 0])

    axes[row, 1] = fig.add_subplot(gs[row, 3], sharex=ref_demand, sharey=ref_demand)
    dens_axes[row, 1] = fig.add_subplot(gs[row, 4], sharey=axes[row, 1])

# Top row: all shapes combined
plot_fan(axes[0, 0], beta_all, "steelblue", line_color="grey", line_alpha=0.06)
plot_density(dens_axes[0, 0], beta_all[:, -1], "steelblue")
plot_fan(axes[0, 1], demand_all, "steelblue", line_color="grey", line_alpha=0.06)
plot_density(dens_axes[0, 1], demand_all[:, -1], "steelblue")

axes[0, 0].set_title(f"PT Preference (u_beta_pt) — all shapes combined (n={len(beta_all)})")
axes[0, 1].set_title(f"Demand growth (u_demand) — all shapes combined (n={len(demand_all)})")

# One row per trajectory shape
for row, shape in enumerate(pw.SHAPES, start=1):
    color = SHAPE_COLORS[shape]
    label = shape.replace("_", " ").title()

    plot_fan(axes[row, 0], beta_by_shape[shape], color)
    plot_density(dens_axes[row, 0], beta_by_shape[shape][:, -1], color)

    plot_fan(axes[row, 1], demand_by_shape[shape], color)
    plot_density(dens_axes[row, 1], demand_by_shape[shape][:, -1], color)

    axes[row, 0].set_title(f"PT Preference (u_beta_pt) — {label} (n={N_PER_SHAPE})")
    axes[row, 1].set_title(f"Demand growth (u_demand) — {label} (n={N_PER_SHAPE})")

for row in range(n_rows):
    axes[row, 0].set_ylabel("u_beta_pt")
    axes[row, 1].set_ylabel("Cumulative demand growth")
    axes[row, 1].yaxis.set_major_formatter(PercentFormatter(1))

axes[-1, 0].set_xlabel("Year")
axes[-1, 1].set_xlabel("Year")
dens_axes[-1, 0].set_xlabel("Density")
dens_axes[-1, 1].set_xlabel("Density")

dens_axes[0, 0].set_title("Final-year\ndistribution", fontsize=9)
dens_axes[0, 1].set_title("Final-year\ndistribution", fontsize=9)

legend_handles = [
    Line2D([0], [0], color="grey", linewidth=1, alpha=0.7,
           label="Individual sampled trajectories"),
    Patch(facecolor="grey", alpha=0.12, label="p10–p90"),
    Patch(facecolor="grey", alpha=0.28, label="p25–p75"),
    Line2D([0], [0], color="black", linewidth=2.2, label="Median"),
    Patch(facecolor="grey", alpha=0.35, label="Final-year density"),
]

fig.suptitle("Trajectory uncertainty by growth shape", y=0.995, fontsize=14)
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.975),
           ncol=5, frameon=False)

fig.subplots_adjust(top=0.94, bottom=0.045, left=0.055, right=0.98)
plt.savefig(PROJECT_ROOT / "figures" / "04_trajectory_fans_by_shape.png",
            dpi=150, bbox_inches="tight")
plt.show()


## State-space trajectories

Each simulated future is a continuous *path* through `(u_demand, u_beta_pt)` space over time — the state in any given year determines that year's transport demand and mode split, while both structural uncertainties evolve simultaneously across the 40-year horizon.


In [ ]:
%matplotlib widget

import mplcursors
from matplotlib.ticker import PercentFormatter

# Generate a reproducible set of 50 futures
N_STATE = 50
years = np.arange(1, N_YEARS + 1)

rng_state = np.random.default_rng(21)
u_beta_state = rng_state.uniform(0, 1, N_STATE)
u_demand_state = rng_state.uniform(0, 1, N_STATE)
shape_beta_state = rng_state.choice(pw.SHAPES, N_STATE)
shape_demand_state = rng_state.choice(pw.SHAPES, N_STATE)

fig, ax = plt.subplots(figsize=(8, 6))
lines, scatters, info = [], [], []

for i, (u_b, u_d, s_b, s_d) in enumerate(zip(
    u_beta_state, u_demand_state,
    shape_beta_state, shape_demand_state
)):
    beta = pw.pt_affinity_trajectory(u_b, s_b)
    demand = pw.demand_growth_trajectory(u_d, s_d)

    line, = ax.plot(demand, beta, color="grey", alpha=0.4,
                    linewidth=1.2, zorder=3)
    scatter = ax.scatter(demand, beta, c=years, cmap="viridis",
                         vmin=years.min(), vmax=years.max(),
                         s=18, alpha=0.8, zorder=2)

    line.set_pickradius(8)
    lines.append(line)
    scatters.append(scatter)
    info.append({
        "u_beta": u_b,
        "u_demand": u_d,
        "beta_shape": s_b,
        "demand_shape": s_d,
    })

cbar = fig.colorbar(scatters[-1], ax=ax)
cbar.set_label("Year")

ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_xlabel("Cumulative demand growth (u_demand)")
ax.set_ylabel("PT Preference Multiplier (u_beta_pt)")
ax.set_title("State-Space Trajectories (Hover over a trajectory to highlight it)")
ax.grid(alpha=0.3)
plt.tight_layout()

# Attach hover interaction to trajectory lines
artist_index = {line: i for i, line in enumerate(lines)}
cursor = mplcursors.cursor(lines, hover=mplcursors.HoverMode.Transient)

@cursor.connect("add")
def highlight(sel):
    selected = artist_index[sel.artist]

    for i, (line, scatter) in enumerate(zip(lines, scatters)):
        active = i == selected
        line.set_alpha(1 if active else 0.04)
        line.set_linewidth(3 if active else 0.6)
        line.set_color("black" if active else "grey")
        scatter.set_alpha(1 if active else 0.03)
        scatter.set_sizes(np.full(N_YEARS, 35 if active else 10))

    d = info[selected]
    sel.annotation.set_text(
        f"Future {selected + 1}\n"
        f"u_beta_pt = {d['u_beta']:.2f} ({d['beta_shape']})\n"
        f"u_demand = {d['u_demand']:.2f} ({d['demand_shape']})"
    )
    sel.annotation.get_bbox_patch().set(fc="white", ec="black", alpha=0.9)
    fig.canvas.draw_idle()

@cursor.connect("remove")
def restore(sel):
    if not cursor.selections:
        for line, scatter in zip(lines, scatters):
            line.set_alpha(0.4)
            line.set_linewidth(1.2)
            line.set_color("grey")
            scatter.set_alpha(0.8)
            scatter.set_sizes(np.full(N_YEARS, 18))
        fig.canvas.draw_idle()

plt.show()




# Part 4 — Trigger rules

## Trigger 1 — Transit Demand Volume (activates Stage 1: Station Package)

| | |
|---|---|
| **Signpost** | `pt_trips` — Peak-hour public transport trips across the corridor |
| **Trigger threshold** | **78,000 trips** (baseline starts around ~71,200 trips) |
| **Must persist?** | Yes — 2 consecutive years above threshold |
| **Decision year** | The year the persistence requirement is met |
| **Lead time** | 2 years (station planning & civil works) |
| **Activation year** | `decision year + 2` |
| **Stage activated** | **Stage 1 (Local Stations & Access Package & Mobility Hubs)** |

## Trigger 2 — Highway Congestion & Delay (activates Stage 2: Core Tunnel)

| | |
|---|---|
| **Signpost** | `avg_tt_min` — Average corridor travel time across passenger trips (minutes) |
| **Trigger threshold** | **17.5 minutes** (highway bottleneck delay reaching critical congestion) |
| **Must persist?** | Yes — 2 consecutive years above threshold |
| **Decision year** | The year the persistence requirement is met |
| **Lead time** | 5 years (tunnel excavation, rolling stock procurement & timetable restructuring) |
| **Activation year** | `decision year + 5` |
| **Stage activated** | **Stage 2 (Core Tunnel & 15-minute Frequency Rhythm)** |

Both triggers evaluate the *previous* year's signpost, never the current one — a real-world investment decision cannot depend on data that has not yet been recorded.

### 4.1 A temporary threshold exceedance that does not trigger
A synthetic example: a scenario that would otherwise stay comfortably below the travel time threshold experiences a one-year highway congestion spike (e.g., due to a major detour, construction event, or bridge repair on the A1 motorway). Average corridor travel time exceeds 17.5 minutes for exactly one year, then reverts to the trendline. Because the rule requires 2 consecutive years of exceedance, this single-year blip is correctly filtered out, preventing the Canton from prematurely committing 2.3 Billion CHF to drill the Core Tunnel based on an anomaly.


In [ ]:
import importlib
import matplotlib.pyplot as plt
import transport_model_interface as tmi
import stages as stages_module
import pathways as pw
import parameters as p

importlib.reload(tmi)
importlib.reload(pw)

if "ctx" not in globals():
    ctx = tmi.load_transport_context(PROJECT_ROOT)
if "corridor_zones" not in globals():
    corridor_zones = tmi.get_zone_ids_for_municipalities(ctx, p.CORRIDOR_MUNICIPALITIES)

# Explicitly load stage specs dict to avoid collision with the imported module name
stage_specs_dict = stages_module.get_stages(p.NOMINAL_PARAMS)

if "STAGE_METRICS" not in globals() or not STAGE_METRICS:
    STAGE_METRICS = {}
    for s in [0, 1, 2]:
        _, metrics = tmi.run_simulation(
            ctx, 
            stage=s, 
            stage_specs=stage_specs_dict, 
            corridor_zone_ids=corridor_zones,
            corridor_municipalities=p.CORRIDOR_MUNICIPALITIES
        )
        STAGE_METRICS[s] = metrics

# Set a moderate baseline where travel time stays below 14.5 min
u_demand = 0.3
u_beta_pt = 0.5
g_traj = pw.demand_growth_trajectory(u_demand, "linear")
pt_traj = pw.pt_affinity_trajectory(u_beta_pt, "linear")

# Use "flexible2" so Stage 1 is already active, allowing us to isolate Trigger 2 perfectly
df_base, meta_base = pw.run_pathway_from_trajectories("flexible2", STAGE_METRICS, g_traj, pt_traj=pt_traj)
max_tt_base = df_base["avg_tt_min"].max()
triggered_base = meta_base["activation2_year"] is not None
print(f"Without the shock: max travel time = {max_tt_base:.2f} min, Trigger 2 fired = {triggered_base}")

# Create a massive one-year traffic demand spike in Year 15
SHOCK_YEAR = 15
g_shock = g_traj.copy()
g_shock[SHOCK_YEAR - 1] += 0.40   

df_shock, meta_shock = pw.run_pathway_from_trajectories("flexible2", STAGE_METRICS, g_shock, pt_traj=pt_traj)
tt_window = df_shock["avg_tt_min"].values[SHOCK_YEAR - 2:SHOCK_YEAR + 1].round(2)
triggered_shock = meta_shock["activation2_year"] is not None
print(f"With a one-year shock in Year {SHOCK_YEAR}: "
      f"travel time in Years {SHOCK_YEAR-1}-{SHOCK_YEAR+1} = {tt_window} min, "
      f"Trigger 2 fired = {triggered_shock}")

years = df_base["year"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(years, df_base["avg_tt_min"], color="steelblue", linewidth=2, label="No shock (Baseline Trend)")
ax.plot(years, df_shock["avg_tt_min"], color="crimson", linewidth=2, label="One-year traffic shock")

ax.axhline(pw.TRIGGER_2["threshold"], color="black", linestyle="--", linewidth=1.2,
           label=f"Trigger 2 threshold ({pw.TRIGGER_2['threshold']} min)")

ax.scatter([SHOCK_YEAR], [df_shock["avg_tt_min"].iloc[SHOCK_YEAR - 1]], color="crimson", zorder=5, s=60)

ax.set_xlabel("Year")
ax.set_ylabel(pw.TRIGGER_2["signpost_label"])
ax.set_title("A one-year exceedance does not meet the 2-year persistence requirement")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_temporary_exceedance.png", dpi=150, bbox_inches="tight")
plt.show()


# Part 5 — Validate the policy behaviour

Before running thousands of futures in the EMA Workbench, we validate the pathway logic on a series of **deterministic, hand-picked scenarios**. We inspect how quickly decisions are taken, how lead times shift activation, and how the transport system responds when major infrastructure (Station Package and Core Tunnel) becomes operational.

For each example:
1. Exogenous corridor travel demand growth (`u_demand`).
2. Trigger 1 signpost: peak-hour public transport demand (`pt_trips` crossing 78,000 trips).
3. Active infrastructure stage over the 40-year horizon (Stage 0, Stage 1, Stage 2).
4. Trigger 2 signpost: highway bottleneck congestion (`avg_tt_min` crossing 17.5 minutes).
5. Core modal performance outcome: resulting public transport mode share (`pt_share`).


In [ ]:
%matplotlib inline

from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

TRIGGER_EXAMPLES = {
    "1. Stable low demand\nNo trigger":
        dict(u_beta=0.5, u_demand=0.05, beta_shape="linear", demand_shape="linear"),
    "2. Rapid early deterioration\nStage 1 triggers early":
        dict(u_beta=0.5, u_demand=0.8, beta_shape="linear", demand_shape="early"),
    "3. Late acceleration\nStage 1 triggers late":
        dict(u_beta=0.5, u_demand=0.4, beta_shape="linear", demand_shape="late"),
    "4. Temporary Year-15 shock\nPersistence test":
        dict(u_beta=0.5, u_demand=0.1, beta_shape="linear", demand_shape="linear", shock_year=15),
    "5. Strong demand and preference\nBoth stages trigger":
        dict(u_beta=0.99, u_demand=0.99, beta_shape="early", demand_shape="early"),
}

def add_event(ax, year, color, style, label=None):
    if year is not None and year <= p.N_YEARS:
        ax.axvline(year, color=color, linestyle=style, linewidth=1.5)
        if label:
            ax.text(year, 0.97, label, color=color, rotation=90,
                    ha="right", va="top", transform=ax.get_xaxis_transform(),
                    fontsize=10)

fig, axes = plt.subplots(len(TRIGGER_EXAMPLES), 5, figsize=(20, 14), sharex=True)

for row, (label, settings) in enumerate(TRIGGER_EXAMPLES.items()):
    kw = settings.copy()
    shock_year = kw.pop("shock_year", None)

    beta_traj = pw.pt_affinity_trajectory(kw["u_beta"], kw["beta_shape"])
    demand_traj = pw.demand_growth_trajectory(kw["u_demand"], kw["demand_shape"])

    if shock_year:
        demand_traj = demand_traj.copy()
        demand_traj[shock_year - 1] += 0.35  # massive 1-year shock to overall demand
        df, meta = pw.run_pathway_from_trajectories(
            "flexible3", STAGE_METRICS, demand_traj, pt_traj=beta_traj
        )
    else:
        df, meta = pw.run_pathway_from_trajectories(
            "flexible3", STAGE_METRICS, demand_traj, pt_traj=beta_traj
        )

    ax0, ax1, ax2, ax3, ax4 = axes[row]
    years = df["year"]

    # 1. Exogenous demand trajectory
    ax0.plot(years, demand_traj, color="navy", linewidth=1.8)
    if shock_year:
        ax0.scatter(shock_year, demand_traj[shock_year - 1],
                    color="crimson", s=45, zorder=4)
    ax0.yaxis.set_major_formatter(PercentFormatter(1))
    ax0.set_ylabel(label, rotation=0, ha="right", va="center",
                   labelpad=15, fontsize=11)

    # 2. Trigger 1: PT Demand Volume (pt_trips > 78,000)
    ax1.plot(df["year"], df["pt_trips"], color="crimson", linewidth=1.8)
    ax1.axhline(pw.TRIGGER_1["threshold"], color="black",
                linestyle="--", linewidth=1.2)
    add_event(ax1, meta["decision1_year"], "royalblue", ":", "D1")
    add_event(ax1, meta["activation1_year"], "royalblue", "--", "A1")

    # 3. Active infrastructure stage
    ax2.step(df["year"], df["stage"], where="post",
             color="steelblue", linewidth=2.2)
    ax2.set_yticks([0, 1, 2])
    ax2.set_yticklabels(["Stage 0 (Base)", "Stage 1 (Stations)", "Stage 2 (Tunnel)"])
    ax2.set_ylim(-0.25, 2.25)
    add_event(ax2, meta["decision1_year"], "royalblue", ":")
    add_event(ax2, meta["activation1_year"], "royalblue", "--")
    add_event(ax2, meta["decision2_year"], "darkorange", ":")
    add_event(ax2, meta["activation2_year"], "darkorange", "--")

    # 4. Trigger 2: Highway Congestion / Delay (avg_tt_min > 17.5 min)
    ax3.plot(df["year"], df["avg_tt_min"], color="darkorange", linewidth=1.8)
    ax3.axhline(pw.TRIGGER_2["threshold"], color="black", linestyle="--", linewidth=1.2)
    add_event(ax3, meta["decision2_year"], "darkorange", ":", "D2")
    add_event(ax3, meta["activation2_year"], "darkorange", "--", "A2")

    # 5. Resulting Public Transport Mode Share
    ax4.plot(df["year"], df["pt_share"] * 100,
             color="seagreen", linewidth=1.8)

    for ax in axes[row]:
        ax.grid(alpha=0.3)

# Column headings
titles = [
    "Demand-growth trajectory",
    f"Trigger 1 signpost\n{pw.TRIGGER_1['signpost_label']}",
    "Active infrastructure stage",
    f"Trigger 2 signpost\n{pw.TRIGGER_2['signpost_label']}",
    "Resulting PT mode share (%)",
]
for ax, title in zip(axes[0], titles):
    ax.set_title(title, fontsize=13, fontweight="bold")

# Axis labels
for ax in axes[-1]:
    ax.set_xlabel("Year")

for row in range(len(TRIGGER_EXAMPLES)):
    axes[row, 1].set_ylabel("PT Trips")
    axes[row, 3].set_ylabel("Avg TT (min)")
    axes[row, 4].set_ylabel("PT Share (%)")

legend_handles = [
    Line2D([0], [0], color="black", linestyle="--",
           label="Trigger threshold"),
    Line2D([0], [0], color="royalblue", linestyle=":",
           label="D1: Stage-1 decision (Stations)"),
    Line2D([0], [0], color="royalblue", linestyle="--",
           label="A1: Stage-1 activation (Stations)"),
    Line2D([0], [0], color="darkorange", linestyle=":",
           label="D2: Stage-2 decision (Tunnel)"),
    Line2D([0], [0], color="darkorange", linestyle="--",
           label="A2: Stage-2 activation (Tunnel)"),
]

fig.suptitle("Flexible 3 — How observed trajectories dynamically activate rail expansions",
             x=0.41, y=0.995, fontsize=16)
fig.legend(handles=legend_handles, loc="upper center",
           bbox_to_anchor=(0.5, 0.968), ncol=5, frameon=False, fontsize=10)

plt.tight_layout(rect=[0.08, 0, 1, 0.945])
plt.savefig(PROJECT_ROOT / "figures" / "04_trigger_examples_combined.png",
            dpi=150, bbox_inches="tight")
plt.show()


Checking that all policy pathways behave properly

We now want to make sure that the policy pathways we've defined behave properly with regards to the triggers and adaptation points. Run this code and select the policy pathway you want to inspect using the drop-down menu.


In [ ]:
%matplotlib inline

import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter


TRIGGER_EXAMPLES = {
    "1. Stable low demand\nNo trigger":
        dict(u_beta=0.5, u_demand=0.05, beta_shape="linear", demand_shape="linear"),
    "2. Rapid deterioration\nEarly Stage 1":
        dict(u_beta=0.5, u_demand=0.8, beta_shape="linear", demand_shape="early"),
    "3. Late acceleration\nLate Stage 1":
        dict(u_beta=0.5, u_demand=0.4, beta_shape="linear", demand_shape="late"),
    "4. Temporary Year-15 shock\nPersistence test":
        dict(u_beta=0.5, u_demand=0.1, beta_shape="linear",
             demand_shape="linear", shock_year=15),
    "5. Strong early growth\nStages 1 and 2":
        dict(u_beta=0.99, u_demand=0.99, beta_shape="early", demand_shape="early"),
}
def add_event(ax, year, color, style, label=None):
    if year is not None and year <= p.N_YEARS:
        ax.axvline(year, color=color, linestyle=style, linewidth=1.5)
        if label:
            ax.text(year, 0.97, label, color=color, rotation=90,
                    ha="right", va="top", fontsize=9,
                    transform=ax.get_xaxis_transform())
def plot_policy_examples(policy):
    spec = pw.PATHWAYS[policy]
    rule1, rule2 = spec["to1"], spec["to2"]
    trigger1 = bool(rule1 and rule1["type"] == "trigger")
    trigger2 = bool(rule2 and rule2["type"] == "trigger")
    fig, axes = plt.subplots(len(TRIGGER_EXAMPLES), 5,
                             figsize=(16, 11), sharex=True)
    for row, (label, settings) in enumerate(TRIGGER_EXAMPLES.items()):
        kw = settings.copy()
        shock_year = kw.pop("shock_year", None)
        beta = pw.pt_affinity_trajectory(kw["u_beta"], kw["beta_shape"])
        demand = pw.demand_growth_trajectory(kw["u_demand"], kw["demand_shape"])
        if shock_year:
            demand = demand.copy()
            demand[shock_year - 1] += 0.35
            df, meta = pw.run_pathway_from_trajectories(policy, STAGE_METRICS, demand, pt_traj=beta)
        else:
            df, meta = pw.run_pathway_from_trajectories(policy, STAGE_METRICS, demand, pt_traj=beta)
        a0, a1, a2, a3, a4 = axes[row]
        d1, a1y = meta["decision1_year"], meta["activation1_year"]
        d2, a2y = meta["decision2_year"], meta["activation2_year"]
        # Demand trajectory
        a0.plot(years, demand, color="navy", linewidth=1.8)
        if shock_year:
            a0.scatter(shock_year, demand[shock_year - 1],
                       color="crimson", s=40, zorder=3)
        a0.yaxis.set_major_formatter(PercentFormatter(1))
        a0.set_ylabel(label, rotation=0, ha="right", va="center",
                      labelpad=12, fontsize=10)
        # Trigger 1 Signpost: PT Trips
        a1.plot(df["year"], df["pt_trips"], color="crimson", linewidth=1.8)
        a1.axhline(pw.TRIGGER_1["threshold"], color="black", linestyle="--", linewidth=1.2)
        add_event(a1, d1, "royalblue", ":", "D1")
        add_event(a1, a1y, "royalblue", "--", "A1")
        # Active stage
        a2.step(df["year"], df["stage"], where="post", color="steelblue", linewidth=2.2)
        a2.set_yticks([0, 1, 2])
        a2.set_yticklabels(["Stage 0 (Base)", "Stage 1 (Stations)", "Stage 2 (Tunnel)"])
        a2.set_ylim(-0.25, 2.25)
        for year, color, style in [(d1, "royalblue", ":"), (a1y, "royalblue", "--"), (d2, "darkorange", ":"), (a2y, "darkorange", "--")]:
            add_event(a2, year, color, style)
        # Trigger 2 Signpost: Travel Time
        a3.plot(df["year"], df["avg_tt_min"], color="darkorange", linewidth=1.8)
        a3.axhline(pw.TRIGGER_2["threshold"], color="black", linestyle="--", linewidth=1.2)
        add_event(a3, d2, "darkorange", ":", "D2")
        add_event(a3, a2y, "darkorange", "--", "A2")
        # Resulting PT Mode Share
        a4.plot(df["year"], df["pt_share"] * 100, color="seagreen", linewidth=1.8)
        a4.axhline(pw.PT_SHARE_TARGET * 100, color="black", linestyle="--", linewidth=1.2)
        for ax in axes[row]:
            ax.grid(alpha=0.25)
            ax.tick_params(labelsize=9)
    titles = [
        "Demand growth",
        f"PT Trips\n{'Trigger 1 signpost' if trigger1 else 'reference threshold'}",
        "Active stage",
        f"Avg TT (min)\n{'Trigger 2 signpost' if trigger2 else 'reference threshold'}",
        "Resulting PT share (%)",
    ]
    for ax, title in zip(axes[0], titles):
        ax.set_title(title, fontsize=12, fontweight="bold")
    for ax in axes[-1]:
        ax.set_xlabel("Year", fontsize=10)
    for row in range(len(TRIGGER_EXAMPLES)):
        axes[row, 1].set_ylabel("PT Trips", fontsize=10)
        axes[row, 3].set_ylabel("Avg TT (min)", fontsize=10)
        axes[row, 4].set_ylabel("PT Share (%)", fontsize=10)
    legend = [Line2D([], [], color="black", linestyle="--", label="Threshold")]
    if rule1:
        if trigger1:
            legend.append(Line2D([], [], color="royalblue", linestyle=":", label="D1: Stage-1 decision (Stations)"))
        legend.append(Line2D([], [], color="royalblue", linestyle="--", label="A1: Stage-1 activation (Stations)"))
    if rule2:
        if trigger2:
            legend.append(Line2D([], [], color="darkorange", linestyle=":", label="D2: Stage-2 decision (Tunnel)"))
        legend.append(Line2D([], [], color="darkorange", linestyle="--", label="A2: Stage-2 activation (Tunnel)"))
    plt.tight_layout(rect=[0.08, 0, 1, 0.93])
    left = axes[0, 0].get_position().x0
    fig.suptitle(f"{spec['name']} — Performance across the same five futures", x=left, y=0.995, ha="left", fontsize=15)
    fig.legend(handles=legend, loc="upper center", bbox_to_anchor=(0.5, 0.965), ncol=len(legend), frameon=False, fontsize=10)
    plt.show()
policy_dropdown = widgets.Dropdown(options=[(pw.PATHWAYS[key]["name"], key) for key in pw.PATHWAY_NAMES], value="flexible3", description="Pathway:")
output = widgets.Output()
def update_policy(change=None):
    with output:
        clear_output(wait=True)
        policy = policy_dropdown.value
        print(pw.PATHWAYS[policy]["description"])
        plot_policy_examples(policy)
policy_dropdown.observe(update_policy, names="value")
display(widgets.VBox([policy_dropdown, output]))
update_policy()


## 5.1 — Comparing all policies under one selected future

Select one public transport preference trajectory (`u_beta_pt`) and one corridor demand-growth trajectory (`u_demand`). The chosen parameter values determine the magnitude of long-term change, while their trajectory shapes dictate when that change occurs.

All nine expansion policies are evaluated against the exact same trajectory path. Consequently, all observed differences in performance, investment timing, and service quality stem directly from the policy rules.


In [ ]:
%matplotlib inline
import ipywidgets as widgets
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

# ------------------------------------------------------------------
# Trajectory Options & Styling
# ------------------------------------------------------------------
BETA_OPTIONS = {
    "Low — linear": (0.15, "linear"), "Central — linear": (0.50, "linear"),
    "High — linear": (0.85, "linear"), "High — early change": (0.85, "early"),
    "High — late change": (0.85, "late"), "High — logistic change": (0.85, "logistic"),
    "Central — random walk": (0.50, "random_walk"), "Central — almost flat": (0.50, "almost_flat"),
}
DEMAND_OPTIONS = {
    "Low — linear": (0.15, "linear"), "Central — linear": (0.50, "linear"),
    "High — linear": (0.85, "linear"), "High — early growth": (0.85, "early"),
    "High — late growth": (0.85, "late"), "High — logistic growth": (0.85, "logistic"),
    "Central — random walk": (0.50, "random_walk"), "Central — almost flat": (0.50, "almost_flat"),
}
POLICY_COLORS = dict(zip(pw.PATHWAY_NAMES, [
    "#7f7f7f", "#4C78A8", "#E45756", "#54A24B", "#72B7B2",
    "#B279A2", "#F2CF5B", "#FF9DA6", "#9D755D"
]))
years_5_1 = np.arange(1, p.N_YEARS + 1)
def compare_policies(beta_choice, demand_choice):
    u_beta, b_shape = BETA_OPTIONS[beta_choice]
    u_demand, d_shape = DEMAND_OPTIONS[demand_choice]
    beta_traj = pw.pt_affinity_trajectory(u_beta, b_shape)
    demand_traj = pw.demand_growth_trajectory(u_demand, d_shape)
    runs = {p_name: pw.run_pathway_from_trajectories(p_name, STAGE_METRICS, demand_traj, pt_traj=beta_traj) for p_name in pw.PATHWAY_NAMES}
    # 3. Metro Map
    fig, ax = plt.subplots(figsize=(11, 4))
    for i, p_name in enumerate(pw.PATHWAY_NAMES):
        df, _ = runs[p_name]
        offset = (i - 4) * 0.03
        ax.step(df["year"], df["stage"] + offset, where="post", lw=2, color=POLICY_COLORS[p_name], label=pw.PATHWAYS[p_name]["name"])
    ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["Stage 0 (Base)", "Stage 1 (Stations)", "Stage 2 (Tunnel)"])
    ax.set_xlim(1, p.N_YEARS); ax.set_ylim(-0.2, 2.2); ax.set_xlabel("Year"); ax.grid(alpha=0.3)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
    ax.set_title("Infrastructure Stage Timing Across Policies", fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show(); plt.close(fig)
    # 4. Detailed 9x7 Matrix Plot
    col_titles = ["PT Trips (T1 Signpost)", "Active Stage", "Avg TT (T2 Signpost)", "Resulting PT Share (%)",
                  "Acceptable Years (%)", "Cumul. TT (Mh)", "Cumul. Cost (MCHF)"]
    ylabels = ["Trips", "Stage", "Avg TT (min)", "PT %", "Accept %", "M hours", "MCHF"]
    thresholds = [pw.TRIGGER_1["threshold"], None, pw.TRIGGER_2["threshold"], pw.PT_SHARE_TARGET * 100, None, None, None]
    fig, axes = plt.subplots(len(pw.PATHWAY_NAMES), 7, figsize=(21, 18), sharex=True, sharey="col")
    for row, p_name in enumerate(pw.PATHWAY_NAMES):
        df, meta = runs[p_name]
        acc = ((df["avg_tt_min"] <= pw.MAX_AVG_TT) & (df["pt_share"] >= pw.PT_SHARE_TARGET)).to_numpy()
        
        series = [
            df["pt_trips"].to_numpy(),             # Col 0: T1 Signpost
            df["stage"].to_numpy(),                # Col 1: Stage
            df["avg_tt_min"].to_numpy(),           # Col 2: T2 Signpost
            df["pt_share"].to_numpy() * 100,       # Col 3: PT share
            np.cumsum(acc) / np.arange(1, p.N_YEARS + 1) * 100,
            df["total_travel_time_hours"].cumsum().to_numpy() / 1e6,
            df["total_cost"].cumsum().to_numpy() / 1e6,
        ]
        events = [(f"D{s}", meta[f"decision{s}_year"], ":", "royalblue" if s == 1 else "darkorange") for s in [1, 2] if meta.get(f"decision{s}_year")] + \
                 [(f"A{s}", meta[f"activation{s}_year"], "--", "royalblue" if s == 1 else "darkorange") for s in [1, 2] if meta.get(f"activation{s}_year")]
        for col, data in enumerate(series):
            ax = axes[row, col]
            if col == 1:
                ax.step(years_5_1, data, where="post", color=POLICY_COLORS[p_name], lw=2)
            else:
                ax.plot(years_5_1, data, color=POLICY_COLORS[p_name], lw=1.8)
            if thresholds[col] is not None:
                ax.axhline(thresholds[col], color="black", ls="--", lw=1.1)
            for tag, yr, ls, clr in events:
                ax.axvline(yr, color=clr, ls=ls, lw=1.2, alpha=0.8)
                if (tag.endswith("1") and col == 0) or (tag.endswith("2") and col == 2):
                    ax.text(yr, 0.95, tag, transform=ax.get_xaxis_transform(), rotation=90, color=clr, fontsize=7, ha="right")
            ax.grid(alpha=0.25); ax.tick_params(labelsize=7); ax.set_ylabel(ylabels[col], fontsize=8)
            if row == 0: ax.set_title(col_titles[col], fontsize=9.5, fontweight="bold")
            if row == len(pw.PATHWAY_NAMES) - 1: ax.set_xlabel("Year", fontsize=8.5)
        axes[row, 0].text(-0.38, 0.5, f"{p_name}", transform=axes[row, 0].transAxes, ha="right", va="center",
                          fontsize=9.5, color=POLICY_COLORS[p_name], fontweight="bold")
        axes[row, 1].set_yticks([0, 1, 2]); axes[row, 1].set_yticklabels(["0", "1", "2"])
    legend = [
        Line2D([], [], color="black", ls="--", label="Threshold"),
        Line2D([], [], color="royalblue", ls=":", label="D1: Stage 1 Decision (Stations)"),
        Line2D([], [], color="royalblue", ls="--", label="A1: Stage 1 Activation (Stations)"),
        Line2D([], [], color="darkorange", ls=":", label="D2: Stage 2 Decision (Tunnel)"),
        Line2D([], [], color="darkorange", ls="--", label="A2: Stage 2 Activation (Tunnel)"),
    ]
    fig.legend(handles=legend, loc="upper center", bbox_to_anchor=(0.55, 0.97), ncol=5, frameon=False, fontsize=9)
    fig.subplots_adjust(top=0.93, bottom=0.04, left=0.12, right=0.99, hspace=0.3, wspace=0.28)
    plt.show(); plt.close(fig)
beta_dropdown = widgets.Dropdown(options=list(BETA_OPTIONS), value="Central — linear", description="PT preference:")
demand_dropdown = widgets.Dropdown(options=list(DEMAND_OPTIONS), value="Central — linear", description="Demand:")
interactive_out = widgets.interactive_output(compare_policies, {"beta_choice": beta_dropdown, "demand_choice": demand_dropdown})
display(widgets.VBox([widgets.HBox([beta_dropdown, demand_dropdown]), interactive_out]))



# Part 6 — Open exploration of all nine policies

We now evaluate all nine expansion policies across the *same* set of sampled futures using the EMA Workbench. Macro-uncertainties (`u_beta_pt`, `u_demand`, `beta_shape`, `demand_shape`) are registered on the `Model`, and `pathway` is registered as the design **lever**. `perform_experiments()` runs the full factorial design across all sampled scenarios.

Two kinds of outcomes are captured from `pathway_ema_model`:
- **`TimeSeriesOutcome`**s (time series of mode shares, travel time, active stage) → organized into **`trajectories_df`** (one row per scenario, policy, year).
- **`ScalarOutcome`**s (Year-40 endpoints and cumulative lifecycle metrics) → organized into **`summary_df`** (one row per scenario, policy) for trade-off analysis and PRIM vulnerability discovery.

A year is defined as **acceptable** if:
1. Average travel time remains within standards: `avg_tt_min <= 20.0 min`, **and**
2. Public transport mode share meets the modal split target: `pt_share >= 35%`.


In [ ]:
TRAJ_COLS = ["car_share", "pt_share", "pt_trips", "avg_tt_min", "total_travel_time_hours", "stage", "co2_tonnes"]

def pathway_ema_model(u_beta_pt=0.5, u_demand=0.5, beta_shape="linear", demand_shape="linear", pathway="baseline", **kwargs):
    """EMA Workbench model function: runs ONE (scenario, policy) pair for the full
    40-year horizon via pw.run_pathway_from_trajectories() with all 16 sampled uncertainties.
    """
    g_traj = pw.demand_growth_trajectory(u_demand, demand_shape)
    pt_traj = pw.pt_affinity_trajectory(u_beta_pt, beta_shape)
    
    # 🎲 Sample all 14 economic/cost nuisance parameters for this specific scenario:
    scenario_params = m.sample_perturbed_params(kwargs)
    
    df, meta = pw.run_pathway_from_trajectories(
        pathway, STAGE_METRICS, g_traj, pt_traj=pt_traj, params=scenario_params
    )
    
    # Check acceptability: travel time <= 20 min and pt_share >= 35%
    acceptable = (df["avg_tt_min"] <= pw.MAX_AVG_TT) & (df["pt_share"] >= pw.PT_SHARE_TARGET)
    out = {col: df[col].values for col in TRAJ_COLS}
    out.update({
        "final_stage": int(df["stage"].iloc[-1]),
        "years_in_stage1": int((df["stage"] == 1).sum()),
        "years_in_stage2": int((df["stage"] == 2).sum()),
        "final_pt_share": float(df["pt_share"].iloc[-1]),
        "final_car_share": float(df["car_share"].iloc[-1]),
        "final_avg_travel_time": float(df["avg_tt_min"].iloc[-1]),
        "cumulative_total_travel_time": float(df["total_travel_time_hours"].sum()),
        "max_travel_time": float(df["avg_tt_min"].max()),
        "min_pt_share": float(df["pt_share"].min()),
        "years_in_failure": int((~acceptable).sum()),
        "pct_acceptable_years": float(acceptable.mean() * 100),
        "total_cost": float(df["total_cost"].sum()),
        "stage1_activation_year": meta["activation1_year"] or 0,
        "stage2_activation_year": meta["activation2_year"] or 0,
        "decision1_year": meta["decision1_year"] or 0,
        "decision2_year": meta["decision2_year"] or 0,
    })
    return out

SCALAR_COLS = [
    "final_stage", "years_in_stage1", "years_in_stage2", "final_pt_share", "final_car_share",
    "final_avg_travel_time", "cumulative_total_travel_time", "max_travel_time", "min_pt_share",
    "years_in_failure", "pct_acceptable_years", "total_cost",
    "stage1_activation_year", "stage2_activation_year", "decision1_year", "decision2_year"
]

pathway_model = Model("AdaptivePathways", function=pathway_ema_model)

# ✅ REGISTER ALL 16 UNCERTAINTIES (2 Structural + 14 Nuisance) + 2 Dynamic Shapes:
pathway_model.uncertainties = (
    m.get_ema_uncertainties(include_nuisance=True)
    + [
        CategoricalParameter("beta_shape", pw.SHAPES),
        CategoricalParameter("demand_shape", pw.SHAPES),
    ]
)
pathway_model.levers = [CategoricalParameter("pathway", pw.PATHWAY_NAMES)]
pathway_model.outcomes = (
    [TimeSeriesOutcome(c) for c in TRAJ_COLS] + [ScalarOutcome(c) for c in SCALAR_COLS]
)

policies = [Policy(name, pathway=name) for name in pw.PATHWAY_NAMES]


N_SCENARIOS = 200 #200 scenarios x 9 policies = 1,800 runs
with SequentialEvaluator(pathway_model) as evaluator:
    ema_experiments, ema_outcomes = perform_experiments(
        pathway_model, N_SCENARIOS, policies=policies, evaluator=evaluator,
        uncertainty_sampling=Samplers.LHS, log_progress=True,
    )

# Format identifiers as clean strings
ema_experiments = ema_experiments.copy()
ema_experiments["scenario"] = ema_experiments["scenario"].astype(int)
for col in ["policy", "pathway", "beta_shape", "demand_shape"]:
    ema_experiments[col] = ema_experiments[col].astype(str)

# All uncertainty columns (u_demand, u_beta_pt, u_C_INV_STAGE1, etc.)
unc_cols = [u.name for u in pathway_model.uncertainties]

n_exp = len(ema_experiments)
trajectories_df = pd.DataFrame({"year": np.tile(np.arange(1, N_YEARS + 1), n_exp)})
for col in TRAJ_COLS:
    trajectories_df[col] = ema_outcomes[col].ravel()
for col in ["scenario", "policy"] + unc_cols:
    trajectories_df[col] = np.repeat(ema_experiments[col].values, N_YEARS)

summary_df = ema_experiments[["scenario", "policy"] + unc_cols].copy()
for col in SCALAR_COLS:
    summary_df[col] = ema_outcomes[col]


scenarios = (ema_experiments[ema_experiments["policy"] == pw.PATHWAY_NAMES[0]]
             [["scenario", "u_beta_pt", "u_demand", "beta_shape", "demand_shape"]]
             .to_dict("records"))

print(f"trajectories_df: {trajectories_df.shape[0]:,} rows ({N_SCENARIOS} scenarios x {len(pw.PATHWAY_NAMES)} policies x {N_YEARS} years)")
print(f"summary_df:      {summary_df.shape[0]:,} rows ({N_SCENARIOS} scenarios x {len(pw.PATHWAY_NAMES)} policies)")

display(summary_df.groupby("policy")[
    ["final_pt_share", "pct_acceptable_years", "cumulative_total_travel_time",
     "max_travel_time", "total_cost", "stage1_activation_year", "stage2_activation_year"]
].mean().round(2).reindex(pw.PATHWAY_NAMES))


# Part 7 — Outcome fan charts

Fan charts (median, interquartile range p25–p75, and 80% interval p10–p90) for four primary transport outcomes: Public Transport Mode Share, Average Travel Time, Total Travel Time, and CO2 Emissions. Outcomes are grouped by policy family: (1) Baseline & Static, (2) Predetermined Staged, and (3) Dynamic Adaptive (Flexible).


In [ ]:
from scipy.stats import gaussian_kde

FAN_OBJECTIVES = [
    ("pt_share", "Public transport modal share", lambda s: s * 100, "%", pw.PT_SHARE_TARGET * 100),
    ("avg_tt_min", "Average travel time", lambda s: s, "min", pw.MAX_AVG_TT),
    ("total_travel_time_hours", "Total travel time", lambda s: s / 1e6, "M hours/year", None),
    ("co2_tonnes", "CO2 emissions", lambda s: s, "tonnes/year", None),
]
POLICY_GROUPS = {
    "Baseline & static": ["baseline", "static1", "static2"],
    "Staged": ["staged1", "staged2", "staged3"],
    "Flexible": ["flexible1", "flexible2", "flexible3"],
}
DENSITY_YEARS = [5, 20, 40]  # early + mid-transition + final-year snapshots

def plot_density(ax, values, color):
    """Thin vertical KDE, sharing its y-axis with the companion fan panel."""
    lo, hi = values.min(), values.max()
    if hi <= lo or np.isnan(lo) or np.isnan(hi):
        return
    y_grid = np.linspace(lo, hi, 200)
    density = gaussian_kde(values)(y_grid)
    ax.fill_betweenx(y_grid, 0, density, color=color, alpha=0.35, zorder=2)
    ax.plot(density, y_grid, color=color, linewidth=1.0, zorder=3)

def fan_chart_grid(policies, group_title):
    n_obj = len(FAN_OBJECTIVES)
    n_dens = len(DENSITY_YEARS)
    cols_per_obj = 1 + n_dens

    fig = plt.figure(figsize=(19 + 2.5 * (n_dens - 1), 4))
    gs = fig.add_gridspec(1, cols_per_obj * n_obj,
                           width_ratios=([7] + [1] * n_dens) * n_obj, wspace=0.08)

    for i, (col, title, scale, unit, threshold) in enumerate(FAN_OBJECTIVES):
        base = cols_per_obj * i
        ax = fig.add_subplot(gs[0, base])
        dens_axes = [fig.add_subplot(gs[0, base + 1 + j], sharey=ax) for j in range(n_dens)]

        for pol in policies:
            sub = trajectories_df[trajectories_df["policy"] == pol]
            g = sub.groupby("year")[col]
            p10, p25, p50, p75, p90 = [scale(g.quantile(q)) for q in (0.10, 0.25, 0.50, 0.75, 0.90)]
            yrs = p50.index
            ax.fill_between(yrs, p10, p90, color=POLICY_COLORS[pol], alpha=0.12,
                            edgecolor=POLICY_COLORS[pol], linewidth=0.4)
            ax.fill_between(yrs, p25, p75, color=POLICY_COLORS[pol], alpha=0.25,
                            edgecolor=POLICY_COLORS[pol], linewidth=0.4)
            ax.plot(yrs, p50, color=POLICY_COLORS[pol], linewidth=2, label=pol)

            for dens_ax, year in zip(dens_axes, DENSITY_YEARS):
                vals = scale(sub.loc[sub["year"] == year, col]).to_numpy()
                plot_density(dens_ax, vals, POLICY_COLORS[pol])

        if threshold is not None:
            ax.axhline(threshold, color="black", linestyle="--", linewidth=1, alpha=0.7)
            for dens_ax in dens_axes:
                dens_ax.axhline(threshold, color="black", linestyle="--", linewidth=1, alpha=0.7)

        for year in DENSITY_YEARS:
            ax.axvline(year, color="grey", linestyle=":", linewidth=0.8, alpha=0.6, zorder=0)

        ax.set_title(f"{title}" + (f" ({unit})" if unit else ""), fontsize=10, fontweight="bold")
        ax.set_xlabel("Year")
        ax.grid(alpha=0.3)

        for dens_ax, year in zip(dens_axes, DENSITY_YEARS):
            dens_ax.set_title(f"Y{year}", fontsize=7, pad=2, color="dimgrey")
            dens_ax.set_xlim(left=0)
            dens_ax.set_xticks([])
            dens_ax.tick_params(axis="y", left=False, labelleft=False)
            for spine in ("top", "right", "left"):
                dens_ax.spines[spine].set_visible(False)

    fig.axes[0].legend(fontsize=8, loc="upper left")
    fig.suptitle(f"{group_title} — Median, p25–p75, p10–p90 "
                 f"(right: density at year {', '.join(str(y) for y in DENSITY_YEARS)})", y=1.06, fontsize=13)
    fig.subplots_adjust(top=0.8, bottom=0.14, left=0.03, right=0.99)
    fname = group_title.lower().replace(" & ", "_").replace(" ", "_")
    plt.savefig(PROJECT_ROOT / "figures" / f"04_fan_{fname}.png", dpi=150, bbox_inches="tight")
    plt.show()

for group_title, policies in POLICY_GROUPS.items():
    fan_chart_grid(policies, group_title)


In [ ]:
%matplotlib inline
import ipywidgets as widgets
from IPython.display import display

def lighten_color(color, factor=0.35):
    import matplotlib.colors as mcolors
    rgb = mcolors.to_rgb(color)
    return [c + (1.0 - c) * factor for c in rgb]

POLICY_LABELS = {
    "baseline": "Baseline", "static1": "Static 1", "static2": "Static 2",
    "staged1": "Staged 1", "staged2": "Staged 2", "staged3": "Staged 3",
    "flexible1": "Flexible 1", "flexible2": "Flexible 2", "flexible3": "Flexible 3",
}

all_policies = pw.PATHWAY_NAMES
final_year = trajectories_df["year"].max()
final_slice = trajectories_df[trajectories_df["year"] == final_year]

# Fixed once from ALL policies so x-axes remain consistent across selections
XLIMS = {}
for col, title, scale, unit, threshold in FAN_OBJECTIVES:
    vals = scale(final_slice[col]).to_numpy()
    v_min, v_max = vals.min(), vals.max()
    v_width = (v_max - v_min) or 1.0
    XLIMS[col] = (v_min - 0.1 * v_width, v_max + 0.1 * v_width)

def plot_ecdf(selected_policies):
    if not selected_policies:
        print("Select at least one policy above.")
        return

    fig, axes = plt.subplots(len(FAN_OBJECTIVES), 1, figsize=(9, 4.5 * len(FAN_OBJECTIVES)))

    for ax, (col, title, scale, unit, threshold) in zip(axes, FAN_OBJECTIVES):
        for pol in selected_policies:
            data = np.sort(scale(final_slice.loc[final_slice["policy"] == pol, col]).to_numpy())
            ecdf_y = np.arange(1, len(data) + 1) / len(data)
            median_val = np.median(data)

            ax.step(data, ecdf_y, where="post", color=POLICY_COLORS[pol], linewidth=2.0,
                    label=f"{POLICY_LABELS[pol]} (median: {median_val:.2f})")
            ax.axvline(median_val, linestyle="--", linewidth=1.3,
                       color=lighten_color(POLICY_COLORS[pol], 0.35))

        if threshold is not None:
            ax.axvline(threshold, color="black", linestyle=":", linewidth=1.8, label=f"Threshold ({threshold})")

        ax.set_xlim(*XLIMS[col])
        ax.set_title(f"{title}" + (f" ({unit})" if unit else "") + f" — Year {final_year}",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel(title + (f" [{unit}]" if unit else ""))
        ax.set_ylabel("Cumulative Probability F(x)")
        ax.set_ylim(0, 1.05)
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.legend(fontsize=8, loc="lower right", frameon=True)

    fig.suptitle(f"Empirical CDFs of Year-{final_year} Outcomes ({len(selected_policies)} selected)",
                 y=1.002, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

policy_select = widgets.SelectMultiple(
    options=[(POLICY_LABELS[p], p) for p in all_policies],
    value=tuple(all_policies),  # all selected by default
    description="Policies:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px", height="170px"),
)

interactive_ecdf = widgets.interactive_output(plot_ecdf, {"selected_policies": policy_select})

display(widgets.VBox([policy_select, interactive_ecdf]))


## FROM HERE ON STILL DIDN'T CHECK CODE - THE CELLS BELOW ARE JUST GENERATED BY AI AFTER I STRUCTURED THE NB04. STILL NEEDS A LOT OF REFINEMENT!!!

## Flexible policies: no / early / medium / late trigger

Within the dynamic adaptive policies (such as `flexible3`), futures naturally group by *when* (if ever) Stage 1 (Station Package) is triggered. We examine the resulting median travel time trajectories across these timing cohorts.


In [ ]:
flex3 = summary_df[summary_df["policy"] == "flexible3"].copy()

def trigger_bucket(row):
    y = row["stage1_activation_year"]
    if y == 0:
        return "No trigger"
    if y <= 10:
        return "Early trigger (≤ Y10)"
    if y <= 25:
        return "Medium trigger (Y11–Y25)"
    return "Late trigger (> Y25)"

flex3["trigger_bucket"] = flex3.apply(trigger_bucket, axis=1)
BUCKET_ORDER = ["No trigger", "Early trigger (≤ Y10)", "Medium trigger (Y11–Y25)", "Late trigger (> Y25)"]
BUCKET_COLORS = {
    "No trigger": "grey",
    "Early trigger (≤ Y10)": "crimson",
    "Medium trigger (Y11–Y25)": "darkorange",
    "Late trigger (> Y25)": "steelblue"
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for bucket in BUCKET_ORDER:
    scenario_ids = flex3.loc[flex3["trigger_bucket"] == bucket, "scenario"]
    if len(scenario_ids) == 0:
        continue
    sub = trajectories_df[(trajectories_df["policy"] == "flexible3") & (trajectories_df["scenario"].isin(scenario_ids))]
    
    # 1. Left Plot: PT Trips (The actual Signpost for Trigger 1)
    med_trips = sub.groupby("year")["pt_trips"].median()
    axes[0].plot(med_trips.index, med_trips.values, color=BUCKET_COLORS[bucket], linewidth=2.2,
                 label=f"{bucket} (n={len(scenario_ids)})")
    
    # 2. Right Plot: Average Travel Time (The downstream performance outcome)
    med_tt = sub.groupby("year")["avg_tt_min"].median()
    axes[1].plot(med_tt.index, med_tt.values, color=BUCKET_COLORS[bucket], linewidth=2.2,
                 label=f"{bucket}")

# Left Panel Details: Trigger 1 Threshold
axes[0].axhline(pw.TRIGGER_1["threshold"], color="black", linestyle="--", linewidth=1.2,
                label=f"Trigger 1 threshold ({pw.TRIGGER_1['threshold']:,} trips)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Peak-Hour PT Demand [trips] (median)")
axes[0].set_title("Stage-1 Signpost: Transit Demand Evolution", fontsize=11, fontweight="bold")
axes[0].legend(fontsize=8.5)
axes[0].grid(alpha=0.3)

# Right Panel Details: Acceptable Max Travel Time
axes[1].axhline(pw.MAX_AVG_TT, color="black", linestyle=":", linewidth=1.2,
                label=f"Max acceptable TT ({pw.MAX_AVG_TT} min)")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Average Travel Time [min] (median)")
axes[1].set_title("Resulting Corridor Travel Time", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=8.5)
axes[1].grid(alpha=0.3)

plt.suptitle("Flexible 3 — System Dynamics by Stage-1 Trigger Timing Cohort", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_flexible_trigger_buckets.png", dpi=150, bbox_inches="tight")
plt.show()

bucket_counts = flex3["trigger_bucket"].value_counts().reindex(BUCKET_ORDER).fillna(0).astype(int)
bucket_summary = pd.DataFrame({
    "n": bucket_counts,
    "%": (bucket_counts / bucket_counts.sum() * 100).round(1),
})
print("Stage 1 (Station Package) Trigger Timing Breakdown:")
display(bucket_summary)


# Part 8 — Parallel-coordinate comparison

Eight key performance metrics across all pathways: final PT mode share, final average travel time, cumulative travel time, maximum travel time, percentage of acceptable years, total lifecycle cost, and the two activation years (0 = never triggered). Each axis is min-max normalized so trade-offs can be compared directly across different units.

Two complementary perspectives:
1. **Full ensemble spread**: Every individual (scenario, policy) run rendered as a semi-transparent trace.
2. **Policy robustness profile**: Median trajectory across all sampled futures to highlight systematic policy behavior.


In [ ]:
PARCOORD_COLS = [
    "final_pt_share", "final_avg_travel_time", "cumulative_total_travel_time",
    "max_travel_time", "pct_acceptable_years", "total_cost",
    "stage1_activation_year", "stage2_activation_year"
]
PARCOORD_LABELS = [
    "Final\nPT share", "Final\ntravel time", "Cumulative\ntravel time",
    "Max\ntravel time", "% acceptable\nyears", "Total\ncost",
    "Stage-1\nactivation yr", "Stage-2\nactivation yr"
]

norm_bounds = {c: (summary_df[c].min(), summary_df[c].max()) for c in PARCOORD_COLS}

def normalize(df):
    out = pd.DataFrame(index=df.index)
    for c in PARCOORD_COLS:
        lo, hi = norm_bounds[c]
        out[c] = (df[c] - lo) / (hi - lo) if hi > lo else 0.5
    return out

norm_all = normalize(summary_df)
x_pos = list(range(len(PARCOORD_COLS)))

# 1. Full ensemble spread
fig, ax = plt.subplots(figsize=(13, 5.5))
for pol in pw.PATHWAY_NAMES:
    idx = summary_df["policy"] == pol
    for _, row in norm_all[idx].iterrows():
        ax.plot(x_pos, row[PARCOORD_COLS].values, color=POLICY_COLORS[pol], alpha=0.06, linewidth=0.7)
for x in x_pos:
    ax.axvline(x, color="lightgray", linewidth=0.8, zorder=0)
ax.set_xticks(x_pos)
ax.set_xticklabels(PARCOORD_LABELS, fontsize=9)
ax.set_ylabel("Normalized value (0 = min, 1 = max)")
ax.set_title(f"Every scenario-policy run across parallel indicators (n={len(summary_df):,})", fontsize=12, fontweight="bold")
handles = [plt.Line2D([0], [0], color=POLICY_COLORS[p], linewidth=2) for p in pw.PATHWAY_NAMES]
ax.legend(handles, pw.PATHWAY_NAMES, loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8.5)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_parcoords_runs.png", dpi=150, bbox_inches="tight")
plt.show()

# 2. Policy-level median robustness
fig, ax = plt.subplots(figsize=(13, 5.5))
for pol in pw.PATHWAY_NAMES:
    med = norm_all[summary_df["policy"] == pol][PARCOORD_COLS].median()
    ax.plot(x_pos, med.values, color=POLICY_COLORS[pol], linewidth=2.5, marker="o", markersize=5, label=pol)
for x in x_pos:
    ax.axvline(x, color="lightgray", linewidth=0.8, zorder=0)
ax.set_xticks(x_pos)
ax.set_xticklabels(PARCOORD_LABELS, fontsize=9)
ax.set_ylabel("Normalized value (0 = min, 1 = max)")
ax.set_title("Policy Robustness Profiles — Median Performance Across All Sampled Futures", fontsize=12, fontweight="bold")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8.5)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_parcoords_policy_summary.png", dpi=150, bbox_inches="tight")
plt.show()


# Part 9 — Trigger and pathway analysis

Focus on the three flexible policies and what actually happens to their triggers across the sampled futures.


In [ ]:
FLEX_POLICIES = ["flexible1", "flexible2", "flexible3"]
flex_summary = summary_df[summary_df["policy"].isin(FLEX_POLICIES)]

prob_rows = []
for pol in FLEX_POLICIES:
    sub = summary_df[summary_df["policy"] == pol]
    prob_rows.append({
        "policy": pol,
        "P(Stage 1 triggered)": (sub["stage1_activation_year"] > 0).mean(),
        "P(Stage 2 triggered)": (sub["stage2_activation_year"] > 0).mean(),
        "P(remains at Stage 0)": (sub["final_stage"] == 0).mean(),
        "P(ends at Stage 1)": (sub["final_stage"] == 1).mean(),
        "P(reaches Stage 2)": (sub["final_stage"] == 2).mean(),
    })
prob_df = pd.DataFrame(prob_rows).set_index("policy")
display((prob_df * 100).round(1))


## Trigger-year distributions and survival curves

We examine the temporal distribution of activation years for triggered scenarios (left) and the "survival curve" showing the fraction of futures that remain at baseline without triggering Stage 1 as time progresses (right).


In [ ]:
FLEX_POLICIES = ["flexible1", "flexible2", "flexible3"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

box_data, box_labels = [], []
for pol in FLEX_POLICIES:
    for col, tag in [("stage1_activation_year", "Stage 1 (Stations)"), ("stage2_activation_year", "Stage 2 (Tunnel)")]:
        vals = summary_df.loc[summary_df["policy"] == pol, col]
        vals = vals[vals > 0]
        if len(vals):
            box_data.append(vals.values)
            box_labels.append(f"{pol}\n{tag}")

axes[0].boxplot(box_data, tick_labels=box_labels)
axes[0].set_ylabel("Activation year (triggered scenarios only)")
axes[0].set_title("Activation-Year Distributions", fontsize=11, fontweight="bold")
axes[0].tick_params(axis="x", labelsize=8)
axes[0].grid(axis="y", alpha=0.3)

for pol in FLEX_POLICIES:
    s1 = summary_df.loc[summary_df["policy"] == pol, "stage1_activation_year"]
    pct_not_yet = [100 * (((s1 == 0) | (s1 > y)).mean()) for y in years]
    axes[1].plot(years, pct_not_yet, color=POLICY_COLORS[pol], linewidth=2, label=f"{pol} (Stage 1)")

axes[1].set_xlabel("Year")
axes[1].set_ylabel("% of scenarios not yet activated")
axes[1].set_title("Survival Curves — Stage 1 (Station Package)", fontsize=11, fontweight="bold")
axes[1].legend(fontsize=8.5)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_trigger_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print("Stage 1 (Station Package) Non-Activation Statistics:")
for pol in FLEX_POLICIES:
    s1 = summary_df.loc[summary_df["policy"] == pol, "stage1_activation_year"]
    n_never = int((s1 == 0).sum())
    print(f"  {pol:10s}: Stage 1 never triggered in {n_never}/{len(s1)} scenarios ({n_never/len(s1):.1%})")


## Stage-occupancy over time

We visualize the evolution of active infrastructure stages across the 40-year horizon for all three flexible policies. The area stackplot shows the proportion of sampled futures operating at Stage 0 (Baseline), Stage 1 (Station Package), and Stage 2 (Core Tunnel) in any given year.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
STAGE_COLORS = {0: "#7f7f7f", 1: "#F2CF5B", 2: "#54A24B"}
STAGE_LABELS = ["Stage 0 (Base)", "Stage 1 (Stations)", "Stage 2 (Tunnel)"]

for ax, pol in zip(axes, FLEX_POLICIES):
    sub = trajectories_df[trajectories_df["policy"] == pol]
    occ = sub.groupby(["year", "stage"]).size().unstack(fill_value=0)
    occ = occ.reindex(columns=[0, 1, 2], fill_value=0)
    occ_pct = occ.div(occ.sum(axis=1), axis=0) * 100
    
    ax.stackplot(
        occ_pct.index,
        [occ_pct[s] for s in [0, 1, 2]],
        colors=[STAGE_COLORS[s] for s in [0, 1, 2]],
        labels=STAGE_LABELS
    )
    ax.set_title(f"{pw.PATHWAYS[pol]['name']}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Year")
    ax.grid(alpha=0.25)
    ax.set_xlim(1, p.N_YEARS)

axes[0].set_ylabel("Share of scenarios (%)")
axes[-1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)

plt.suptitle("Stage Occupancy Over Time — Flexible Policies", y=1.04, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_stage_occupancy.png", dpi=150, bbox_inches="tight")
plt.show()


## Trigger timing by uncertainty trajectory pattern
We cross-tabulate Stage 1 (Station Package) trigger timing against the structural trajectory shapes for public transport preference (`beta_shape`) and demand growth (`demand_shape`). The heatmaps show the mean activation year (left) and the percentage of futures where Stage 1 is never triggered (right).

In [ ]:
flex3_s = summary_df[summary_df["policy"] == "flexible3"].copy()
flex3_s["s1_triggered"] = flex3_s["stage1_activation_year"].where(flex3_s["stage1_activation_year"] > 0)

heat = flex3_s.pivot_table(index="beta_shape", columns="demand_shape", values="s1_triggered",
                            aggfunc="mean").reindex(index=pw.SHAPES, columns=pw.SHAPES)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

sns.heatmap(heat, annot=True, fmt=".1f", cmap="YlOrRd", ax=axes[0],
            cbar_kws={"label": "Mean Stage-1 activation year"})
axes[0].set_title("Flexible 3 — Mean Stage-1 Activation Year\n(NaN = never triggered)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Demand growth shape (demand_shape)")
axes[0].set_ylabel("PT preference shape (beta_shape)")

never_rate = flex3_s.assign(never=lambda d: d["stage1_activation_year"] == 0).pivot_table(
    index="beta_shape", columns="demand_shape", values="never", aggfunc="mean"
).reindex(index=pw.SHAPES, columns=pw.SHAPES) * 100

sns.heatmap(never_rate, annot=True, fmt=".0f", cmap="Blues", ax=axes[1],
            cbar_kws={"label": "% of scenarios never triggered"})
axes[1].set_title("Flexible 3 — % Never Triggered Stage 1", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Demand growth shape (demand_shape)")
axes[1].set_ylabel("PT preference shape (beta_shape)")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_trigger_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## Stage 1 vs. Stage 2 activation, and time spent in Stage 1 (Flexible 3)


In [ ]:
flex3_both = flex3_s[(flex3_s["stage1_activation_year"] > 0)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
triggered2 = flex3_both["stage2_activation_year"] > 0
axes[0].scatter(flex3_both.loc[triggered2, "stage1_activation_year"],
                 flex3_both.loc[triggered2, "stage2_activation_year"],
                 color="crimson", alpha=0.7, s=30, label="Both triggered")
axes[0].scatter(flex3_both.loc[~triggered2, "stage1_activation_year"],
                 [N_YEARS + 2] * (~triggered2).sum(),
                 color="lightgray", alpha=0.7, s=30, marker="x", label="Stage 2 never triggered")
axes[0].plot([0, N_YEARS], [0, N_YEARS], color="black", linestyle=":", linewidth=1)
axes[0].set_xlabel("Stage-1 activation year"); axes[0].set_ylabel("Stage-2 activation year")
axes[0].set_title("Flexible 3 — Stage-1 vs. Stage-2 activation"); axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].hist(flex3_s["years_in_stage1"], bins=20, color="#F2CF5B", edgecolor="white")
axes[1].set_xlabel("Years spent in Stage 1"); axes[1].set_ylabel("Number of scenarios")
axes[1].set_title("Flexible 3 — time spent in Stage 1")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_flexible3_stage_timing.png", dpi=150, bbox_inches="tight")
plt.show()


# Part 10 — Time-series clustering

Do corridor average travel time trajectories under **Flexible 3** fall into a handful of recurring *dynamic shapes*? 

Hierarchical clustering groups trajectories by how similar they look across the entire 40-year time series (Euclidean distance between the 40-year travel time vectors, Ward linkage) — rather than by their Year-40 value alone. Two futures that end up at the same final travel time but experienced different trigger timings (e.g. an early capacity crunch vs. a late surge) land in distinct clusters.




In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import numpy as np

# 1. Pivot 40-year average travel time trajectories for Flexible 3 across all 200 scenarios
flex3_traj = trajectories_df[trajectories_df["policy"] == "flexible3"].sort_values(["scenario", "year"])
pivot_tt = flex3_traj.pivot(index="scenario", columns="year", values="avg_tt_min")
X_tt = pivot_tt.values
scenario_order = pivot_tt.index  # scenario IDs matching row order

# 2. Hierarchical clustering using Ward's minimum variance method
Z = linkage(X_tt, method="ward")
n_leaves = len(scenario_order)

CLUSTER_COLORS = ["#4C78A8", "#E45756", "#54A24B", "#F2CF5B", "#B279A2", "#72B7B2", "#9D755D", "#FF9DA6"]
DEFAULT_LINK_COLOR = "lightgray"

def cluster_color(c):
    return CLUSTER_COLORS[(c - 1) % len(CLUSTER_COLORS)]

def link_color_dict(Z, cluster_labels, n_leaves):
    """Dendrogram branch colors matching fcluster labels."""
    leaf_colors = {i: cluster_color(cluster_labels[i]) for i in range(n_leaves)}
    link_colors = {}
    for i, (a, b, dist, n) in enumerate(Z):
        a, b = int(a), int(b)
        node_id = i + n_leaves
        ca = link_colors.get(a, leaf_colors.get(a))
        cb = link_colors.get(b, leaf_colors.get(b))
        link_colors[node_id] = ca if ca == cb else DEFAULT_LINK_COLOR
    return link_colors

def plot_cluster_trajectories(cluster_labels, title, fname):
    n_clusters = cluster_labels.max()
    total = len(cluster_labels)
    fig, ax = plt.subplots(figsize=(10, 5))
    for c in range(1, n_clusters + 1):
        scen_ids = scenario_order[cluster_labels == c]
        sub = flex3_traj[flex3_traj["scenario"].isin(scen_ids)]
        color = cluster_color(c)
        for _, grp in sub.groupby("scenario"):
            ax.plot(grp["year"], grp["avg_tt_min"], color=color, alpha=0.15, linewidth=0.7, zorder=1)
        med = sub.groupby("year")["avg_tt_min"].median()
        pct = 100 * len(scen_ids) / total
        ax.plot(med.index, med.values, color=color, linewidth=2.4, zorder=3,
                label=f"Cluster {c} (n={len(scen_ids)}, {pct:.0f}%)")
    
    # Use max acceptable travel time standard (15 min) as reference line
    ax.axhline(pw.MAX_AVG_TT, color="black", linestyle="--", linewidth=1.2, alpha=0.8, 
               label=f"Max Acceptable Travel Time ({pw.MAX_AVG_TT} min)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Corridor Average Travel Time (min)")
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "figures" / fname, dpi=150, bbox_inches="tight")
    plt.show()


# ------------------------------------------------------------------
# A. Elbow Plot — Ward merge distance vs. number of clusters
# ------------------------------------------------------------------
last_rev = Z[:, 2][::-1]
n_show = min(20, len(last_rev))
idxs = np.arange(1, n_show + 1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(idxs, last_rev[:n_show], marker="o", color="steelblue", linewidth=2)
ax.set_xlabel("Number of clusters"); ax.set_ylabel("Ward merge distance")
ax.set_title("Flexible 3 — Elbow Plot (Ward Linkage on Travel Time Trajectories)")
ax.set_xticks(idxs)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_cluster_elbow.png", dpi=150, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------------
# B. Cut dendrogram at k=4 clusters and map scenario IDs -> cluster IDs
# ------------------------------------------------------------------
N_CLUSTERS = 4
cluster_labels = fcluster(Z, t=N_CLUSTERS, criterion="maxclust")
cluster_map = dict(zip(scenario_order, cluster_labels))  # Maps scenario_id -> cluster_id (1..4)

plot_cluster_trajectories(
    cluster_labels,
    f"Flexible 3 — Corridor Travel Time Trajectories by Cluster (k={N_CLUSTERS})",
    "04_cluster_trajectories.png",
)

# ------------------------------------------------------------------
# C. Dendrogram Plot
# ------------------------------------------------------------------
link_colors = link_color_dict(Z, cluster_labels, n_leaves)
fig, ax = plt.subplots(figsize=(13, 4.5))
dendrogram(Z, ax=ax, no_labels=True,
           link_color_func=lambda k: link_colors.get(k, DEFAULT_LINK_COLOR))
ax.set_title(f"Dendrogram — Flexible 3 Travel Time Trajectories (k={N_CLUSTERS})")
ax.set_xlabel("Scenario"); ax.set_ylabel("Ward distance")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_cluster_dendrogram.png", dpi=150, bbox_inches="tight")
plt.show()


# Part 11 — Explain the trajectory clusters

For each trajectory cluster: we analyze the underlying uncertainty drivers (`u_beta_pt`, `u_demand`, trajectory shapes) and how they translate into downstream outcomes (trigger timing, years in failure, final PT mode share, and cumulative travel time).


In [ ]:
flex3_summary = summary_df[summary_df["policy"] == "flexible3"].copy()
if "cluster_map" in globals():
    flex3_summary["cluster"] = flex3_summary["scenario"].map(cluster_map)

cluster_profile = flex3_summary.groupby("cluster").agg(
    n=("scenario", "size"),
    mean_u_beta_pt=("u_beta_pt", "mean"),
    mean_u_demand=("u_demand", "mean"),
    p_stage1_triggered=("stage1_activation_year", lambda s: (s > 0).mean() * 100),
    p_stage2_triggered=("stage2_activation_year", lambda s: (s > 0).mean() * 100),
    mean_stage1_year=("stage1_activation_year", lambda s: s[s > 0].mean()),
    mean_years_in_failure=("years_in_failure", "mean"),
    mean_pct_acceptable=("pct_acceptable_years", "mean"),
    mean_final_pt_share=("final_pt_share", lambda s: s.mean() * 100),
    mean_cumulative_tt=("cumulative_total_travel_time", lambda s: s.mean() / 1e6),
)

CLUSTER_NAMES = {
    1: "Escalating demand, early trigger",
    2: "Low pressure, no trigger",
    3: "Moderate growth, late trigger",
    4: "Sustained high transit adoption",
}

# Apply cluster names if available, else keep numeric cluster index
cluster_profile.index = [CLUSTER_NAMES.get(c, f"Cluster {c}") for c in cluster_profile.index]
cluster_profile = cluster_profile.rename(columns={
    "mean_u_beta_pt": "Mean u_beta_pt",
    "mean_u_demand": "Mean u_demand",
    "p_stage1_triggered": "P(Stage 1) %",
    "p_stage2_triggered": "P(Stage 2) %",
    "mean_stage1_year": "Mean S1 year",
    "mean_years_in_failure": "Failure years",
    "mean_pct_acceptable": "Acceptable %",
    "mean_final_pt_share": "Final PT share %",
    "mean_cumulative_tt": "Cumul TT (Mh)",
})
display(cluster_profile.round(2))

N_CLUSTERS = len(cluster_profile)
for c in range(1, N_CLUSTERS + 1):
    sub = flex3_summary[flex3_summary["cluster"] == c]
    if len(sub):
        b_counts = sub["beta_shape"].value_counts().to_dict()
        d_counts = sub["demand_shape"].value_counts().to_dict()
        c_name = CLUSTER_NAMES.get(c, f"Cluster {c}")
        print(f"{c_name}: beta_shape={b_counts} | demand_shape={d_counts}")


## Which uncertain factors predict cluster membership?

We use an **ExtraTreesClassifier** feature importance model to quantify which uncertain parameters (`u_demand`, `u_beta_pt`) and trajectory shapes most strongly predict which cluster a scenario will fall into.


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

feature_cols = ["u_beta_pt", "u_demand"]
X_features = flex3_summary[feature_cols].copy()
for col in ["beta_shape", "demand_shape"]:
    dummies = pd.get_dummies(flex3_summary[col], prefix=col, dtype=int)
    X_features = pd.concat([X_features, dummies], axis=1)

clf = ExtraTreesClassifier(n_estimators=250, random_state=0)
clf.fit(X_features, flex3_summary["cluster"])
importance = pd.Series(clf.feature_importances_, index=X_features.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
importance.plot(kind="barh", ax=ax, color="seagreen")
ax.invert_yaxis()
ax.set_xlabel("Feature importance (normalized score)")
ax.set_title("Which Uncertain Factors Predict Trajectory Cluster Membership?", fontsize=11, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_cluster_feature_scores.png", dpi=150, bbox_inches="tight")
plt.show()

print("Feature importance measures PREDICTIVE relevance for cluster membership, not causality.")


# Part 12 — Adaptation tipping points

**Adaptation tipping point:** the first year the *current* infrastructure stage — held fixed, on its own, for the entire 40-year horizon — can no longer satisfy the acceptable corridor performance requirement (`avg_tt_min <= 20.0` min and `pt_share >= 35.0%`). 

This is an intrinsic property of a **stage and a future**, not of any policy: it asks *"how long would Stage $k$ alone have been sufficient in this future,"* independent of whether any policy actually upgrades it.

We compute tipping points for Stage 0 (Baseline Network), Stage 1 (Station Package), and Stage 2 (15-min frequency upgrade) separately, across the same 200 sampled scenarios from Part 6.


In [ ]:
import importlib
import pathways as pw
importlib.reload(pw)

# ==============================================================================
# Part 12.1 — Compute Tipping Points Across All Futures
# ==============================================================================

stage_specs = stages_module.get_stages(p.NOMINAL_PARAMS)

tipping_rows = []
for sc in scenarios:
    g_traj = pw.demand_growth_trajectory(sc["u_demand"], sc["demand_shape"])
    tipping_rows.append({
        **sc,
        "tipping_stage0": pw.find_tipping_point(0, STAGE_METRICS, g_traj),
        "tipping_stage1": pw.find_tipping_point(1, STAGE_METRICS, g_traj),
        "tipping_stage2": pw.find_tipping_point(2, STAGE_METRICS, g_traj),
    })
tipping_df = pd.DataFrame(tipping_rows)

STAGE_LABELS = {0: "Stage 0 (Baseline)", 1: "Stage 1 (Stations)", 2: "Stage 2 (Tunnel)"}

for stage in [0, 1, 2]:
    col = f"tipping_stage{stage}"
    never = tipping_df[col].isna().mean() * 100
    med = tipping_df[col].median()
    label = STAGE_LABELS[stage]
    print(f"{label}: never fails within 40y in {never:.0f}% of futures"
          + (f"; median tipping year (when it does fail) = Year {med:.0f}" if not np.isnan(med) else ""))


In [ ]:
# ==============================================================================
# Part 12.2 — Tipping-Point Distributions & Survival Curves
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
STAGE_TIP_COLORS = {0: "#7f7f7f", 1: "#F2CF5B", 2: "#54A24B"}

for stage in [0, 1, 2]:
    vals = tipping_df[f"tipping_stage{stage}"].dropna()
    if len(vals):
        axes[0].hist(vals, bins=range(1, 42, 2), alpha=0.6, color=STAGE_TIP_COLORS[stage], 
                     label=f"{STAGE_LABELS[stage]} (n={len(vals)})")
axes[0].set_xlabel("Tipping-point year")
axes[0].set_ylabel("Count of futures")
axes[0].set_title("Tipping-Point Year Distributions")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

for stage in [0, 1, 2]:
    tvals = tipping_df[f"tipping_stage{stage}"]
    pct_still_ok = [100 * ((tvals.isna()) | (tvals > y)).mean() for y in years]
    axes[1].plot(years, pct_still_ok, color=STAGE_TIP_COLORS[stage], linewidth=2.2, label=STAGE_LABELS[stage])
axes[1].set_xlabel("Year")
axes[1].set_ylabel("% of futures still sufficient")
axes[1].set_title("Survival Curves — How Long Each Stage Remains Sufficient")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_tipping_points.png", dpi=150, bbox_inches="tight")
plt.show()


**How long does Stage 0 normally remain sufficient?**
Essentially never (Year 1 in ~100% of futures). The baseline railway corridor already exceeds the 20.0-minute average travel time threshold or fails the 35% PT share requirement at current passenger volumes without dedicated bypass tunnel capacity.

**In which futures is Stage 1 sufficient until Year 40?**
In low-demand growth futures (`almost_flat` demand trajectory or low `u_demand`). Deploying the Station Package & Mobility Hubs provides local access relief; if regional demand grows slowly, this capacity headroom lasts the entire 40-year horizon without needing the Core Tunnel.

**In which futures is Stage 2 required?**
Whenever demand growth accelerates significantly (`early`, `logistic`, or high `u_demand`), causing Stage 1's tipping point to occur before Year 40. In those futures, the baseline tracks and station packages alone become congested, requiring Stage 2 (Brüttenertunnel Core Tunnel & 15-minute frequency rhythm) to sustain acceptable travel times.


In [ ]:
# ==============================================================================
# Part 12.3 — Stage 1 Tipping Point by Trajectory Pattern
# ==============================================================================
heat_tip = tipping_df.pivot_table(index="beta_shape", columns="demand_shape", values="tipping_stage1",
                                   aggfunc="median").reindex(index=pw.SHAPES, columns=pw.SHAPES)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.heatmap(heat_tip, annot=True, fmt=".0f", cmap="YlOrRd_r", ax=ax,
            cbar_kws={"label": "Median Stage-1 tipping-point year"})
ax.set_title("Stage-1 Tipping Point by Trajectory Pattern\n(blank = never fails within 40y)")
ax.set_xlabel("demand_shape")
ax.set_ylabel("beta_shape (PT preference)")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_tipping_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


# Part 13 — Opportunity points

**Opportunity point:** the first year upgrading to the *next* stage becomes clearly worthwhile — achieving travel time relief of at least 2.0 minutes on the corridor — even if the current stage has not technically crossed its 20.0-minute failure threshold yet. 

This is deliberately distinct from a tipping point:
- A **tipping point** asks: *"Has the current stage already failed?"* (reactive threshold)
- An **opportunity point** asks: *"Would switching now already yield significant benefit?"* (proactive investment window)


In [ ]:
# ==============================================================================
# Part 13.1 — Opportunity Points vs. Tipping Points
# ==============================================================================
opportunity_rows = []
for sc in scenarios:
    g_traj = pw.demand_growth_trajectory(sc["u_demand"], sc["demand_shape"])
    opportunity_rows.append({
        **sc,
        "opportunity_0to1": pw.find_opportunity_point(0, 1, STAGE_METRICS, g_traj, benefit_margin=2.0),
        "opportunity_1to2": pw.find_opportunity_point(1, 2, STAGE_METRICS, g_traj, benefit_margin=2.0),
    })
opportunity_df = pd.DataFrame(opportunity_rows)
combo_df = tipping_df.merge(opportunity_df[["scenario", "opportunity_0to1", "opportunity_1to2"]], on="scenario")

n_never_needed = combo_df["tipping_stage0"].isna().sum()
n_stage2_needed = (combo_df["tipping_stage1"].notna()).sum()
n_stage0_tips = combo_df["tipping_stage0"].notna().sum()
n_can_postpone = (combo_df["opportunity_0to1"] > combo_df["tipping_stage0"]).sum()

print(f"Futures where Stage 1 is never needed (Stage 0 never tips): {n_never_needed}/{len(combo_df)}")
print(f"Futures where Stage 2 eventually becomes necessary (Stage 1 tips): {n_stage2_needed}/{len(combo_df)}")
print(f"Futures where Stage 1 opportunity point comes AFTER Stage 0 tipping point: {n_can_postpone}/{n_stage0_tips}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for col, color, label in [("opportunity_0to1", "seagreen", "Opportunity: Stage 0 -> 1 (Station Package)"),
                          ("tipping_stage0", "crimson", "Tipping: Stage 0 (Baseline)")]:
    vals = combo_df[col].dropna()
    ax.hist(vals, bins=range(1, 42, 2), alpha=0.5, color=color, label=f"{label} (n={len(vals)})")
ax.set_xlabel("Year")
ax.set_ylabel("Count of futures")
ax.set_title("Opportunity Point vs. Tipping Point — Stage 0 -> Stage 1 (Station Package)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_opportunity_vs_tipping.png", dpi=150, bbox_inches="tight")
plt.show()


# Part 14 — Trigger timing vs. tipping points

Compare each flexible policy's *actual* activation year against the tipping point of the stage it is replacing — the **lead margin**:

$$\text{lead margin} = \text{tipping-point year} - \text{activation year}$$

- **Positive ($> 0$):** Timely adaptation — the intervention arrives *before* the corridor suffers unacceptable travel times.
- **Zero ($= 0$):** Exactly at the wire.
- **Negative ($< 0$):** Late adaptation — the corridor operates in a congested failure state before the upgrade opens.


In [ ]:
# ==============================================================================
# Part 14.1 — Lead Margin Classification Across Flexible Policies
# ==============================================================================
FLEX_POLICIES = ["flexible1", "flexible2", "flexible3"]

def classify_lead_margin(tipping_year, activation_year):
    if pd.isna(tipping_year) and pd.isna(activation_year):
        return "adaptation not needed"
    if pd.isna(tipping_year) and pd.notna(activation_year):
        return "unnecessarily early adaptation"
    if pd.notna(tipping_year) and pd.isna(activation_year):
        return "missed adaptation"
    margin = tipping_year - activation_year
    return "timely adaptation" if margin >= 0 else "late adaptation"

flex_lead = summary_df[summary_df["policy"].isin(FLEX_POLICIES)].merge(
    tipping_df[["scenario", "tipping_stage0", "tipping_stage1"]], on="scenario"
)
flex_lead["activation1"] = flex_lead["stage1_activation_year"].replace(0, np.nan)
flex_lead["activation2"] = flex_lead["stage2_activation_year"].replace(0, np.nan)
flex_lead["lead_margin_1"] = flex_lead["tipping_stage0"] - flex_lead["activation1"]
flex_lead["lead_margin_2"] = flex_lead["tipping_stage1"] - flex_lead["activation2"]
flex_lead["class_1"] = flex_lead.apply(lambda r: classify_lead_margin(r["tipping_stage0"], r["activation1"]), axis=1)

print("Stage 1 (Station Package) adaptation classification, by flexible policy (% of scenarios):")
display((flex_lead.groupby(["policy", "class_1"]).size().unstack(fill_value=0)
         .div(flex_lead.groupby("policy").size(), axis=0) * 100).round(1))


In [ ]:
# ==============================================================================
# Part 14.2 — Activation vs. Tipping Point Scatter & Lead-Margin Distributions
# ==============================================================================
PATHWAY_COLORS = {
    "flexible1": "crimson",
    "flexible2": "darkorange",
    "flexible3": "steelblue"
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# 1. Scatter: Stage-0 Tipping vs Stage-1 Activation
for pol in ["flexible1", "flexible3"]:
    sub = flex_lead[(flex_lead["policy"] == pol) & flex_lead["tipping_stage0"].notna() & flex_lead["activation1"].notna()]
    axes[0].scatter(sub["tipping_stage0"], sub["activation1"], alpha=0.6, s=25,
                    color=PATHWAY_COLORS.get(pol, "grey"), label=pol)
axes[0].plot([0, N_YEARS], [0, N_YEARS], color="black", linestyle=":", linewidth=1, label="on time")
axes[0].set_xlabel("Stage-0 tipping-point year")
axes[0].set_ylabel("Stage-1 activation year")
axes[0].set_title("Activation vs. Tipping Point (Stage 0 -> Stage 1)")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# 2. Boxplot: Lead-Margin Distributions
margin_data = [flex_lead.loc[flex_lead["policy"] == pol, "lead_margin_1"].dropna() for pol in ["flexible1", "flexible3"]]
axes[1].boxplot(margin_data, tick_labels=["flexible1", "flexible3"])
axes[1].axhline(0, color="black", linestyle="--", linewidth=1, label="Tipping threshold")
axes[1].set_ylabel("Lead margin (years) [positive = in time]")
axes[1].set_title("Stage 1 Lead-Margin Distributions")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_lead_margin.png", dpi=150, bbox_inches="tight")
plt.show()


# Part 15 — Critical points in outcome trajectories

For a single multi-decade travel time trajectory, several "critical year" definitions coexist and are important to distinguish in infrastructure planning:

- **Strongest annual change:** The year with the largest single-year drop/jump in corridor travel time (typically the year the Brüttenertunnel or 15-min rhythm is commissioned).
- **Strongest change in slope:** The year the trajectory's curvature changes most (acceleration in passenger demand growth).
- **First year of failure:** The first year corridor performance falls below acceptable thresholds (`avg_tt_min > 20.0` min or `pt_share < 35.0%`).
- **Intervention year:** When a capacity upgrade is completed and commissioned (the *activation* year).
- **Recovery year:** The first year *after* an initial failure period that acceptable travel times are restored.

> **Key Distinction:** A sharp step-down in travel time when a new tunnel opens is a **policy-induced change point** (the planned intervention taking effect), whereas the **first year of failure** reflects the underlying system dynamic (demand outpacing baseline track capacity).


In [ ]:
# ==============================================================================
# Part 15.1 — Detect Critical Points on Flexible 3 Trajectories
# ==============================================================================
def critical_points(series, acceptable_mask):
    diffs = np.diff(series)
    d2 = np.diff(diffs)
    strongest_change_year = int(np.argmax(np.abs(diffs))) + 2       # +2: diff index 0 -> year 2
    strongest_slope_year = int(np.argmax(np.abs(d2))) + 3 if len(d2) else None
    failing = np.where(~acceptable_mask)[0]
    first_failure_year = int(failing[0]) + 1 if len(failing) else None
    recovery_year = None
    if len(failing):
        after = np.where(acceptable_mask[failing[0]:])[0]
        if len(after):
            recovery_year = int(failing[0] + after[0]) + 1
    return dict(strongest_change_year=strongest_change_year, strongest_slope_year=strongest_slope_year,
                first_failure_year=first_failure_year, recovery_year=recovery_year)

# Representative corridor growth futures
representative_futures = {
    "Low Demand Growth (Baseline sufficient)": {"u_demand": 0.2, "demand_shape": "almost_flat"},
    "Moderate Growth (Tunnel triggered mid-term)": {"u_demand": 0.5, "demand_shape": "linear"},
    "High Demand Surge (Dual triggers: Tunnel + 15-min)": {"u_demand": 0.85, "demand_shape": "early"},
}

critical_rows = []
for label, kw in representative_futures.items():
    df, meta = pw.run_pathway("flexible3", STAGE_METRICS, **kw)
    acceptable = ((df["avg_tt_min"] <= pw.MAX_AVG_TT) & (df["pt_share"] >= pw.PT_SHARE_TARGET)).values
    cp = critical_points(df["avg_tt_min"].values, acceptable)
    critical_rows.append({
        "Future": label, **cp,
        "decision1_year": meta["decision1_year"], "activation1_year": meta["activation1_year"],
        "decision2_year": meta["decision2_year"], "activation2_year": meta["activation2_year"],
        "years_in_failure": int((~acceptable).sum()),
    })


display(pd.DataFrame(critical_rows).set_index("Future"))


# Part 16 — Scenario discovery with PRIM on dynamic failures

We apply the **Patient Rule Induction Method (PRIM)** to discover which combinations of deep uncertainties drive corridor vulnerability over time.

We define a **dynamic failure** across the 40-year horizon as:
- Fewer than half the years acceptable (`pct_acceptable_years < 50%`), **or**
- Average travel time at Year 40 exceeds the threshold (`final_avg_travel_time > 20.0` min), **or**
- Public transport mode share fails to reach the 35% corridor target (`final_pt_share < 35.0%`).

We run PRIM separately for **Baseline** (no project) and **Flexible 3** (adaptive pathway) across the two core structural uncertainties: `u_beta_pt` (rail preference) and `u_demand` (demand growth).


In [ ]:
# ==============================================================================
# Part 16.1 — Define Dynamic Failure & Run PRIM
# ==============================================================================
import matplotlib.patches as mpatches
from ema_workbench.analysis import prim

summary_df["dynamic_failure"] = (
    (summary_df["pct_acceptable_years"] < 50)
    | (summary_df["final_avg_travel_time"] > pw.MAX_AVG_TT)
    | (summary_df["final_pt_share"] < pw.PT_SHARE_TARGET)
)

for pol in ["baseline", "flexible3"]:
    rate = summary_df.loc[summary_df["policy"] == pol, "dynamic_failure"].mean()
    print(f"{pol}: dynamic failure rate = {rate:.0%}")

prim_results = {}
for pol in ["baseline", "flexible3"]:
    sub = summary_df[summary_df["policy"] == pol]
    x = sub[["u_beta_pt", "u_demand"]]
    y = sub["dynamic_failure"].values

    fail_rate = y.mean()
    if fail_rate in (0.0, 1.0):
        prim_results[pol] = None
        verdict = "fails" if fail_rate == 1.0 else "succeeds"
        print(f"{pol}: {fail_rate:.0%} failure rate — entire space {verdict} (uniform outcome, no PRIM box needed).\n")
        continue

    prim_alg = prim.Prim(x, y, peel_alpha=0.1)
    box = prim_alg.find_box()
    prim_results[pol] = (prim_alg, box)
    box.show_tradeoff()
    plt.title(f"PRIM Peeling Trajectory — {pol}")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "figures" / f"04_prim_tradeoff_{pol}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"{pol}: coverage={box.coverage:.0%}  density={box.density:.0%}  mass={box.mass:.0%}")
    print(prim_alg.boxes_to_dataframe())
    print()


In [ ]:
# ==============================================================================
# Part 16.2 — Visual Comparison of Vulnerability Boxes
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pol in zip(axes, ["baseline", "flexible3"]):
    sub = summary_df[summary_df["policy"] == pol]
    ax.scatter(sub.loc[~sub["dynamic_failure"], "u_demand"], sub.loc[~sub["dynamic_failure"], "u_beta_pt"],
               color="lightgray", s=20, alpha=0.8, label="Acceptable")
    ax.scatter(sub.loc[sub["dynamic_failure"], "u_demand"], sub.loc[sub["dynamic_failure"], "u_beta_pt"],
               color="crimson", s=20, alpha=0.8, label="Dynamic failure")
    ax.set_xlabel("u_demand (Demand growth draw)")
    ax.set_ylabel("u_beta_pt (PT affinity draw)")

    if prim_results[pol] is None:
        fail_rate = sub["dynamic_failure"].mean()
        ax.set_title(f"{pol.capitalize()}\n{fail_rate:.0%} failure — unmitigated vulnerability")
    else:
        box = prim_results[pol][1]
        box1 = prim_results[pol][0].boxes_to_dataframe()["box 1"]

        def bound(dim):
            return (float(box1.loc[dim, "min"]), float(box1.loc[dim, "max"])) if dim in box1.index else (0.0, 1.0)
        beta_lo, beta_hi = bound("u_beta_pt")
        demand_lo, demand_hi = bound("u_demand")
        ax.add_patch(mpatches.Rectangle((demand_lo, beta_lo), demand_hi - demand_lo, beta_hi - beta_lo,
                                         fill=False, edgecolor="black", linewidth=2, linestyle="--"))
        ax.set_title(f"{pol.capitalize()}\ncoverage={box.coverage:.0%}, density={box.density:.0%}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Vulnerability Region: Baseline Network vs. Adaptive Pathway (Flexible 3)", y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_prim_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


### Key Insights from Scenario Discovery

1. **Baseline Failure (100%):** Without major capacity expansion, existing corridor tracks fail in every sampled future due to unavoidable bottleneck congestion.
2. **Vulnerability Reduction:** **Flexible 3** dramatically contracts the failure region, restricting residual vulnerability to futures with weak public transport affinity (`low u_beta_pt`) combined with rapid demand growth (`high u_demand`).
3. **The Adaptation Lag:** Even under an adaptive policy, some failure years occur early on while waiting for the 2-year persistence check and construction lead time before the Brüttenertunnel becomes operational.


# Part 17 — Dynamic adaptive-pathways map

We synthesize the simulation results into an empirical **Adaptive Pathways Map**: across all 200 sampled futures under **Flexible 3**, which infrastructure pathway was actually realized, and at what point in time?


In [ ]:
# ==============================================================================
# Part 17.1 — Empirical Realization of Adaptive Pathways (Flexible 3)
# ==============================================================================
STAGE_TIP_COLORS = {0: "#7f7f7f", 1: "#F2CF5B", 2: "#54A24B"}

flex3_paths = summary_df[summary_df["policy"] == "flexible3"].copy()
flex3_paths["route"] = np.select(
    [flex3_paths["final_stage"] == 0, flex3_paths["final_stage"] == 1, flex3_paths["final_stage"] == 2],
    ["Stayed at Stage 0 (Baseline)", "Stage 1 only (Station Package)", "Reached Stage 2 (Core Tunnel & 15-min)"],
    default="Unknown",
)
route_counts = flex3_paths["route"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(11, 5))
ax.axhline(0, color=STAGE_TIP_COLORS[0], linewidth=6, alpha=0.3, solid_capstyle="butt")

s1_years = flex3_paths.loc[flex3_paths["stage1_activation_year"] > 0, "stage1_activation_year"]
s2_years = flex3_paths.loc[flex3_paths["stage2_activation_year"] > 0, "stage2_activation_year"]

# Scatter of actual activation years
ax.scatter(s1_years, np.ones(len(s1_years)) * 0, color="#F2CF5B", s=20, alpha=0.6, zorder=3, label="Stage 1 activation")
for y in [s1_years.quantile(q) for q in (0.1, 0.5, 0.9)]:
    ax.axvline(y, color="#F2CF5B", alpha=0.4, linestyle="--", linewidth=1.2)

if len(s2_years):
    ax.scatter(s2_years, np.ones(len(s2_years)) * 1, color="#54A24B", s=20, alpha=0.6, zorder=3, label="Stage 2 activation")
    for y in [s2_years.quantile(q) for q in (0.1, 0.5, 0.9)]:
        ax.axvline(y, color="#54A24B", alpha=0.4, linestyle="--", linewidth=1.2)

ax.plot([1, N_YEARS], [-0.3, -0.3], color="black", alpha=0)
ax.set_yticks([0, 1])
ax.set_yticklabels(["Activate Stage 1\n(Station Package)", "Activate Stage 2\n(Core Tunnel)"])
ax.set_xlabel("Year")
ax.set_xlim(0, N_YEARS + 1)
ax.set_title("Flexible 3 — Realized Trigger Timing Across Sampled Futures")
ax.grid(axis="x", alpha=0.3)

# Summary box on the right
textstr = "\n".join([f"{route}: {pct:.1f}%" for route, pct in route_counts.items()])
ax.text(1.02, 0.5, textstr, transform=ax.transAxes, fontsize=9.5, va="center",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="grey"))

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "04_pathways_map_data.png", dpi=150, bbox_inches="tight")
plt.show()

print("Realized Pathway Shares:")
print(route_counts.round(1))
if len(s1_years):
    print(f"\nStage 1 (Station Package) activation timing (triggered futures): "
          f"p10 = Year {s1_years.quantile(0.1):.0f}, median = Year {s1_years.median():.0f}, p90 = Year {s1_years.quantile(0.9):.0f}")
if len(s2_years):
    print(f"Stage 2 (Core Tunnel) activation timing (triggered futures): "
          f"p10 = Year {s2_years.quantile(0.1):.0f}, median = Year {s2_years.median():.0f}, p90 = Year {s2_years.quantile(0.9):.0f}")


### Key Insights on Dynamic Pathways

1. **Pathway Branching:** Fixed strategies (`static1`, `staged1`) commit to a single rigid construction timeline regardless of real-world demand. In contrast, **Flexible 3** naturally branches: low-demand futures remain at Stage 1 without wasting billions on unnecessary frequency expansions, while high-growth futures advance to Stage 2.
2. **Timing Spread:** The dispersion in activation years (shown by the spread of dots and the 10th/50th/90th percentile dashed lines) highlights how flexibility tailors capital investment to the exact trajectory of corridor growth.
3. **Core Planning Trade-off:** Adaptive pathways maximize long-term economic efficiency and avoid stranded railway assets, in exchange for less upfront certainty in long-range parliamentary budget schedules.


# Part 18 — Planner-in-the-loop interactive demonstration

In this interactive demonstration, you act as the cantonal and federal railway planning team. 

A **hidden future** is drawn at random — you do not know `u_beta_pt`, `u_demand`, or their trajectory shapes in advance. You only observe what real planners track over time: the corridor travel time (`avg_tt_min`), rail mode share (`pt_share`), the active infrastructure stage, and construction lead times.

At decision checkpoints (Years **1, 5, 10, 15, 20, 25, 30, 35**), you choose whether to:
1. **Wait** (preserve financial options and monitor demand trends)
2. **Implement Stage 1 (Station Package)** (2-year construction lead time)
3. **Implement Stage 2 (Core Tunnel & 15-min rhythm frequency expansion)** (5-year construction lead time; available once Stage 1 is active)

At Year 40, the hidden future is revealed and your pathway is evaluated against the 9 predefined benchmark policies.


In [ ]:
# ==============================================================================
# Part 18.1 — Interactive Planner-in-the-Loop Simulation
# ==============================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output

CHECKPOINTS = [1, 5, 10, 15, 20, 25, 30, 35, N_YEARS]

STAGE_NAMES = {
    0: "Stage 0 (Baseline Network)", 
    1: "Stage 1 (Connecting Stations A3, A4, A5)", 
    2: "Stage 2 (Core Tunnel & Winterthur Hub A0, A1, A2)"
}

def init_game(seed=None):
    rng_hidden = np.random.default_rng(seed)
    u_b = float(rng_hidden.uniform(0, 1))
    u_d = float(rng_hidden.uniform(0, 1))
    b_shape = rng_hidden.choice(pw.SHAPES)
    d_shape = rng_hidden.choice(pw.SHAPES)
    
    return {
        "u_beta_pt": u_b,
        "u_demand": u_d,
        "beta_shape": b_shape,
        "demand_shape": d_shape,
        "g_traj": pw.demand_growth_trajectory(u_d, d_shape),
        "pt_traj": pw.pt_affinity_trajectory(u_b, b_shape),
        "stage": 0,
        "pending_stage1_year": None,
        "pending_stage2_year": None,
        "history": [],
        "decisions": [],
        "year": 0,
        "checkpoint_idx": 0,
    }

def simulate_up_to(target_year):
    for t in range(game_state["year"], target_year):
        year = t + 1
        if game_state["pending_stage1_year"] == year:
            game_state["stage"] = max(game_state["stage"], 1)
        if game_state["pending_stage2_year"] == year:
            game_state["stage"] = max(game_state["stage"], 2)
            
        stage = game_state["stage"]
        base_m = STAGE_METRICS.get(stage, {})
        g_val = game_state["g_traj"][t]
        row = m.simulate_year(base_m, stage, t, g_val, p.NOMINAL_PARAMS)

        total_daily_trips = base_m.get("total_trips", 0) * (1 + g_val)
        
        # Apply modal preference trajectory to pt_share
        base_pt = base_m.get("pt_share", 0.0)
        pt_affinity = game_state["pt_traj"][t]
        pt_share = base_pt * pt_affinity

        # Upfront option fees and capital investments
        inv_this_year = p.C_FLEX if (stage > 0 and year == 1) else 0.0
        if game_state["pending_stage1_year"] == year:
            inv_this_year += p.C_INV_STAGE1
        if game_state["pending_stage2_year"] == year:
            inv_this_year += p.C_INV_STAGE2
        op_this_year = {0: 0.0, 1: p.C_OP_STAGE1, 2: p.C_OP_STAGE1 + p.C_OP_STAGE2}[stage]

        row.update({
            "year": year,
            "stage": stage,
            "avg_tt_min": row.get("avg_tt_min", 0.0),
            "pt_share": pt_share,
            "total_travel_time_hours": row.get("avg_tt_min", 0.0) * total_daily_trips / 60,
            "inv_cost": inv_this_year,
            "op_cost": op_this_year,
            "total_cost": (
                row.get("car_cost", 0.0)
                + row.get("pt_cost", 0.0)
                + inv_this_year
                + op_this_year
            ),
        })
        game_state["history"].append(row)
    game_state["year"] = target_year

def apply_decision(choice):
    current_year = game_state["year"]
    game_state["decisions"].append((current_year, choice))
    if "Stage 1" in choice and game_state["pending_stage1_year"] is None:
        game_state["pending_stage1_year"] = current_year + pw.TRIGGER_1["lead_time"]
    elif "Stage 2" in choice and game_state["pending_stage2_year"] is None:
        game_state["pending_stage2_year"] = current_year + pw.TRIGGER_2["lead_time"]
    game_state["checkpoint_idx"] += 1
    if game_state["checkpoint_idx"] < len(CHECKPOINTS):
        simulate_up_to(CHECKPOINTS[game_state["checkpoint_idx"]])

def get_history_df():
    return pd.DataFrame(game_state["history"])

# Initialize game state
game_state = init_game(seed=None)
simulate_up_to(CHECKPOINTS[0])
print(f"Hidden future generated. Decision checkpoints at Years: {CHECKPOINTS[:-1]} (Year {N_YEARS} reveal).")

output_area = widgets.Output()
button_box = widgets.HBox([])

def render_state():
    with output_area:
        clear_output(wait=True)
        df = get_history_df()
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
        
        # Left: Average Travel Time
        axes[0].plot(df["year"], df["avg_tt_min"], color="crimson", linewidth=2)
        axes[0].axhline(pw.MAX_AVG_TT, color="black", linestyle="--", linewidth=1, label=f"Max TT ({pw.MAX_AVG_TT} min)")
        axes[0].set_xlim(1, N_YEARS)
        axes[0].set_ylabel("Corridor Avg Travel Time (min)")
        axes[0].set_xlabel("Year")
        axes[0].legend(fontsize=8)
        axes[0].grid(alpha=0.3)
        
        # Right: Public Transport Mode Share
        axes[1].plot(df["year"], df["pt_share"] * 100, color="forestgreen", linewidth=2)
        axes[1].axhline(pw.PT_SHARE_TARGET * 100, color="black", linestyle="--", linewidth=1, label=f"PT Target ({pw.PT_SHARE_TARGET*100:.0f}%)")
        axes[1].set_xlim(1, N_YEARS)
        axes[1].set_ylabel("Rail / PT Share (%)")
        axes[1].set_xlabel("Year")
        axes[1].legend(fontsize=8)
        axes[1].grid(alpha=0.3)
        
        status_str = f"Year {game_state['year']} — Active Stage: {STAGE_NAMES[game_state['stage']]}"
        if game_state["pending_stage1_year"] and game_state["pending_stage1_year"] > game_state["year"]:
            status_str += f" | Station Package opening in Year {game_state['pending_stage1_year']}"
        if game_state["pending_stage2_year"] and game_state["pending_stage2_year"] > game_state["year"]:
            status_str += f" | Core Tunnel opening in Year {game_state['pending_stage2_year']}"
            
        plt.suptitle(status_str, fontsize=11, fontweight="bold")
        plt.tight_layout()
        plt.show()
        print("Decisions taken so far:", game_state["decisions"])
    render_buttons()

def render_buttons():
    if game_state["checkpoint_idx"] >= len(CHECKPOINTS) - 1:
        button_box.children = []
        with output_area:
            print(f"\nReached Year {N_YEARS}. Run the scorecard cell below to reveal the future and compare performance.")
        return
    options = ["Wait"]
    if game_state["stage"] < 1 and game_state["pending_stage1_year"] is None:
        options.append("Implement Stage 1 (Local Stations & Access)")
    if game_state["stage"] >= 1 and game_state["stage"] < 2 and game_state["pending_stage2_year"] is None:
        options.append("Implement Stage 2 (Core Tunnel & 15-min rhythm)")
    buttons = []
    for opt in options:
        b = widgets.Button(description=opt, layout=widgets.Layout(width="280px"))
        def on_click(_, opt=opt):
            apply_decision(opt)
            render_state()
        b.on_click(on_click)
        buttons.append(b)
    button_box.children = buttons

render_state()
display(widgets.VBox([output_area, button_box]))


## Scorecard

Run this once you've made all your decisions (or at any point to see where you'd stand if the game ended now).


In [ ]:
# ==============================================================================
# Scorecard: Planner Pathway vs. Benchmark Policies
# ==============================================================================
def scorecard_for(df_eval):
    acceptable = (df_eval["avg_tt_min"] <= pw.MAX_AVG_TT) & (df_eval["pt_share"] >= pw.PT_SHARE_TARGET)
    return {
        "final_stage": int(df_eval["stage"].iloc[-1]),
        "total_cost_MCHF": df_eval["total_cost"].sum() / 1e6,
        "cumulative_tt_hours": df_eval["total_travel_time_hours"].sum(),
        "max_avg_tt_min": df_eval["avg_tt_min"].max(),
        "years_in_failure": int((~acceptable).sum()),
        "pct_acceptable_years": acceptable.mean() * 100,
        "stage2_avoided": bool(df_eval["stage"].iloc[-1] < 2),
    }

student_df = get_history_df()
student_card = scorecard_for(student_df)

print(f"HIDDEN FUTURE REVEALED: u_beta_pt = {game_state['u_beta_pt']:.2f} ({game_state['beta_shape']}), "
      f"u_demand = {game_state['u_demand']:.2f} ({game_state['demand_shape']})\n")

predefined_cards = {}
for pol in pw.PATHWAY_NAMES:
    df_pol, meta_pol = pw.run_pathway_from_trajectories(
        pol, STAGE_METRICS, game_state["g_traj"], params=p.NOMINAL_PARAMS, pt_traj=game_state["pt_traj"]
    )
    predefined_cards[pol] = scorecard_for(df_pol)

scoreboard = pd.DataFrame({"PLANNER PATHWAY": student_card, **predefined_cards}).T
best_predefined = scoreboard.drop(index="PLANNER PATHWAY")["total_cost_MCHF"].idxmin()
scoreboard["regret_vs_best_MCHF"] = scoreboard["total_cost_MCHF"] - scoreboard.loc[best_predefined, "total_cost_MCHF"]

display(scoreboard.round(2))

your_regret = scoreboard.loc["PLANNER PATHWAY", "regret_vs_best_MCHF"]
print(f"Best benchmark policy for this specific hidden future (by lowest total cost): {best_predefined}")
print(f"Planner pathway regret vs. best policy: {your_regret:.1f} MCHF")


The scorecard deliberately keeps every indicator **separate** — cost, travel time, congestion, failure years, and regret each tell a different part of the story, and folding them into one unexplained number would hide exactly the trade-offs this notebook has spent so long making visible.


In [ ]:
#total runtime notebook04
print(f"__NOTEBOOK_RUNTIME_SECONDS__={_time.perf_counter() - _NB_START:.3f}")
print(f"__NOTEBOOK_RUNTIME_MINUTES__={(_time.perf_counter() - _NB_START) / 60:.3f}")
